# Day Ahead Plan Max Utility

Run from `Code/`, with the input datasets in `../Data/`. Requires NumPy, pandas, Matplotlib, openpyxl, Pyomo, HiGHS and IPython. Figures and tables are displayed in the notebook; no images, result files or solver logs are exported.

The workflow is configuration, auxiliary functions, preprocessing, characterization, day-ahead planning, reporting, and integrated real-time control. Cross-strategy figures run the other planning notebooks in isolated memory, using the same block duration; these additional solves can take several minutes.


In [ ]:
import openpyxl
import pyomo.environ as pe
import numpy as np
import os
import pandas as pd
from datetime import datetime
from datetime import timedelta
import matplotlib.pyplot as plt
from dataclasses import dataclass
import itertools
from matplotlib.patches import Patch

In [ ]:
# ============================================================
# GLOBAL TIME AND LIMIT SETTINGS
# ============================================================

BLOCK_SIZE = 30
N_BLOCKS = 24 * 60 // BLOCK_SIZE
TOTAL_MINUTES = 24 * 60
INCOMPLETE_POLICY = "observed"

# ============================================================
# TIME ZONES
# ============================================================
#
# 0 -> 00:00 - 09:59
# 1 -> 10:00 - 19:29
# 2 -> 19:30 - 22:29
# 3 -> 22:30 - 23:59
#
# start and end are expressed in minutes
# ============================================================

TIME_ZONES = {
    0: {
        "start": 0,
        "end": 600
    },

    1: {
        "start": 600,
        "end": 1170
    },

    2: {
        "start": 1170,
        "end": 1350
    },

    3: {
        "start": 1350,
        "end": 1440
    }
}


# ============================================================
# FIXED POWER LIMITS BY TIMEZONE [W]
# Shared by Day-Ahead and the integrated Real-Time controller.
# ============================================================
TIMEZONE_POWER_LIMITS_W = {
    0: 2500.0,
    1: 13000.0,
    2: 16000.0,
    3: 7500.0,
}


# ============================================================
# POWER LIMITS
# PV + BESS scenario
# ============================================================

timezone_power_limits_W_PV_BESS = {
    0: 4000.0,
    1: 8000.0,
    2: 12000.0,
    3: 6000.0
}
timezone_energy_limits_Wh_PV_BESS = {
    tz: (
        P_lim
        * BLOCK_SIZE
        / 60.0
    )
    for tz, P_lim
    in timezone_power_limits_W_PV_BESS.items()
}

# ============================================================
# 3-ToU ENERGY PRICE VECTOR
# ============================================================

TOU_PRICE_EUR_KWH = {
    0: 0.15,   # Off-peak: 00:00 - 10:00
    1: 0.25,   # Shoulder: 10:00 - 19:30
    2: 0.45,   # On-peak: 19:30 - 22:30
    3: 0.15,   # Off-peak: 22:30 - 24:00
}


# Auxiliary functions

In [ ]:
"""Day-ahead solver diagnostics retained in memory.

Execution time includes APPSI model transfer and solve, but excludes reporting.
Physical metrics use unrounded solver decisions; energy and relaxation are kWh.
The legacy APPSI ``solution.gap`` is an absolute gap and is deliberately not used.
"""


import math
import re
import time
import warnings
from datetime import datetime, timezone
import contextlib
import io

import pyomo
import pyomo.environ as pe


SOLUTION_METRICS = (
    "objective_value", "energy_consumption_kWh", "energy_cost_EUR",
    "relaxation_kWh", "time_deviation_cost", "power_deviation_cost",
    "total_utility_objective",
)


def _number(value):
    """Missing/undefined/infinite solver quantities are explicitly NaN."""
    try:
        number = float(value)
        return number if math.isfinite(number) else math.nan
    except (ValueError, TypeError):
        return math.nan








def skipped_diagnostics(scenario_name, reason, *, time_limit=60, mip_rel_gap=0.01):
    """Record a skipped dependent model without inventing a solver run or log."""
    record = dict.fromkeys(SOLUTION_METRICS, math.nan)
    record.update(
        scenario_name=scenario_name, time_limit_s=time_limit,
        mip_rel_gap_target=mip_rel_gap, execution_time_s=math.nan,
        termination_condition="not_run", feasible_solution=False,
        optimality_proven=False, primal_bound=math.nan, dual_bound=math.nan,
        mip_gap_pct=math.nan, run_executed=False, diagnostic_note=reason,
        log_file="", log_complete=False,
    )
    return record


def day_ahead_operational_metrics(model, *, Pmax, DC_FC, E_NF, price, block_size):
    """Reuse the notebooks' physical block-energy formula (W * minutes / 60).

    ``energy_cost`` in Min Consumption is actually energy, so it cannot be
    used as a common electricity-cost metric. Likewise, relaxation_cost has
    different units across objectives; always sum the E_rlx variables in Wh.
    """
    energy_Wh = {}
    for k in model.K:
        if hasattr(model, "energy_consumption"):
            energy_Wh[k] = pe.value(model.energy_consumption[k])
        else:
            energy_Wh[k] = (
                E_NF[k]
                + sum(Pmax[j] * pe.value(model.n_on[j, k]) / 60.0
                      for j in model.JT)
                + sum(Pmax[j] * pe.value(model.Y_rec[j, k]) * DC_FC[j, k]
                      * block_size / 60.0 for j in model.JP)
            )
    metrics = {
        "energy_consumption_kWh": sum(energy_Wh.values()) / 1000.0,
        "energy_cost_EUR": sum(price[k] * energy_Wh[k] / 1000.0 for k in model.K),
        "relaxation_kWh": sum(pe.value(model.E_rlx[k]) for k in model.K) / 1000.0,
    }
    if hasattr(model, "time_deviation_cost"):
        metrics.update(
            time_deviation_cost=pe.value(model.time_deviation_cost),
            power_deviation_cost=pe.value(model.power_deviation_cost),
            total_utility_objective=pe.value(model.objective),
        )
    return metrics


def _log_metric(log_text, label, *, percent=False):
    suffix = r"%" if percent else r"(?:\s|$)"
    matches = re.findall(r"^\s*" + re.escape(label) + r"\s+([^\s%]+)" + suffix,
                         log_text, flags=re.MULTILINE)
    return _number(matches[-1]) if matches else math.nan


def solve_day_ahead(model, scenario_name, time_limit=60, mip_rel_gap=0.01, tee=True,
                    *, operational_metrics=None):
    """Solve with legacy appsi_highs and safely load only a feasible incumbent.

    Native HiGHS info provides the unrounded relative gap. Access to APPSI's
    internal highspy handle is isolated and optional: legacy bounds and native
    log summaries provide a fallback if that handle changes in future versions.
    No gap is inferred from the objective value or the legacy absolute gap.
    """
    solver = pe.SolverFactory("appsi_highs")
    solver.highs_options = {
        "time_limit": time_limit,
        "mip_rel_gap": mip_rel_gap,
        "output_flag": True,
        "log_to_console": tee,
        "log_file": "",
    }
    # `tee` is an APPSI solve argument, not a valid HiGHS option.
    started = time.perf_counter()
    log_buffer = io.StringIO()
    with contextlib.redirect_stdout(log_buffer):
        result = solver.solve(model, tee=True, load_solutions=False)
    log_text = log_buffer.getvalue()
    if tee:
        print(log_text, end="")
    elapsed = time.perf_counter() - started

    objective = next(model.component_data_objects(pe.Objective, active=True))
    minimizing = objective.sense == pe.minimize
    # APPSI inserts a legacy solution only when best_feasible_objective exists.
    # This also avoids accidentally treating values inherited by a clone as a
    # new incumbent when the second solve finds no solution.
    feasible = len(result.solution) > 0
    primal = _number(result.problem.upper_bound if minimizing
                     else result.problem.lower_bound) if feasible else math.nan
    dual = _number(result.problem.lower_bound if minimizing
                   else result.problem.upper_bound)
    gap_pct = math.nan
    gap_source = "unavailable"
    highs_status = "unavailable"
    highs_version = "unavailable"
    native = getattr(solver, "_solver_model", None)
    if native is not None:
        try:
            info = native.getInfo()
            highs_version = native.version()
            highs_status = native.modelStatusToString(native.getModelStatus())
            if getattr(info, "valid", False):
                if feasible:
                    primal = _number(info.objective_function_value)
                # mip_dual_bound/mip_gap do not describe LP solves.
                if getattr(info, "mip_node_count", -1) >= 0:
                    dual = _number(info.mip_dual_bound)
                    if feasible:
                        gap_pct = 100.0 * _number(info.mip_gap)
                        if math.isfinite(gap_pct):
                            gap_source = "HiGHS getInfo().mip_gap"
        except (AttributeError, TypeError, RuntimeError) as error:
            warnings.warn(f"Native HiGHS metrics unavailable: {error}")
        finally:
            # Ensure that native logging remains disabled.
            native.setOptionValue("log_file", "")

    log_complete = (
        "Solving report" in log_text
        and all(label in log_text for label in ("Status", "Primal bound", "Dual bound", "Gap"))
    ) or ("Model status" in log_text and "HiGHS run time" in log_text)
    if not log_complete:
        warnings.warn("HiGHS console output lacks a final summary.")
    if feasible and not math.isfinite(primal):
        primal = _log_metric(log_text, "Primal bound")
    if not math.isfinite(dual):
        dual = _log_metric(log_text, "Dual bound")
    if feasible and not math.isfinite(gap_pct):
        gap_pct = _log_metric(log_text, "Gap", percent=True)
        if math.isfinite(gap_pct):
            gap_source = "HiGHS final log (rounded)"

    diagnostics = dict.fromkeys(SOLUTION_METRICS, math.nan)
    diagnostics.update(
        scenario_name=scenario_name, time_limit_s=time_limit,
        mip_rel_gap_target=mip_rel_gap, execution_time_s=elapsed,
        termination_condition=str(result.solver.termination_condition),
        feasible_solution=feasible,
        optimality_proven=bool(feasible and pe.check_optimal_termination(result)),
        primal_bound=primal, dual_bound=dual, mip_gap_pct=gap_pct,
        gap_source=gap_source, objective_sense="minimize" if minimizing else "maximize",
        run_executed=True, log_file="", log_complete=log_complete,
        pyomo_version=pyomo.__version__, highs_version=highs_version,
        highs_status=highs_status, timestamp_utc=datetime.now(timezone.utc).isoformat(),
    )
    if feasible:
        # Supported APPSI method; no automatic optimality requirement.
        solver.load_vars()
        diagnostics["objective_value"] = pe.value(objective)
        if operational_metrics is not None:
            diagnostics.update(operational_metrics(model))

    print(f"{scenario_name}: {diagnostics['termination_condition']}; "
          f"feasible={feasible}; optimality proven={diagnostics['optimality_proven']}; "
          f"gap={gap_pct:.6g}%")
    return solver, result, diagnostics


In [ ]:
def run_day_ahead_for_comparison(notebook_name, block_size):
    """Execute planning in an isolated namespace; return fresh in-memory tables.

    Stop at the planning-results tag, before cross-strategy comparisons and RT.
    No saved results or notebook outputs are read. Each call performs new solves.
    """
    import contextlib
    import io
    import json
    import re
    from pathlib import Path
    from IPython.display import display

    allowed = {
        "Day_Ahead_Plan_Min_Cost.ipynb",
        "Day_Ahead_Plan_Min_Consumption.ipynb",
        "Day_Ahead_Plan_Max_Utility.ipynb",
    }
    if notebook_name not in allowed or block_size not in (10, 15, 30):
        raise ValueError("Select a day-ahead notebook and B=10, 15, or 30.")
    notebook = json.loads(Path(notebook_name).read_text())
    scope = {"__name__": "comparison_run", "display": lambda *args, **kwargs: None}
    existing_figures = set(plt.get_fignums())
    try:
        with contextlib.redirect_stdout(io.StringIO()), plt.ioff():
            for index, item in enumerate(notebook["cells"]):
                if item["cell_type"] != "code":
                    continue
                source = "".join(item["source"])
                source = re.sub(r"^BLOCK_SIZE\s*=.*$", f"BLOCK_SIZE = {block_size}", source, flags=re.M)
                exec(compile(source, f"{notebook_name}:cell{index}", "exec"), scope)
                if "planning-results" in item.get("metadata", {}).get("tags", []):
                    if not scope["day_ahead_feasible"]:
                        raise RuntimeError(f"No feasible planning result: {notebook_name}")
                    return scope
        raise ValueError(f"Missing planning-results tag: {notebook_name}")
    finally:
        for number in set(plt.get_fignums()) - existing_figures:
            plt.close(number)


In [ ]:
def plot_power_flexible_behavior(
    schedule_results_df,
    device,
    block_size=15,
    tick_step=120
):
    """Compare recommended and forecast power fractions per block."""
    import numpy as np
    import matplotlib.pyplot as plt

    if block_size <= 0 or tick_step <= 0:
        raise ValueError("block_size and tick_step must be positive.")
    df_plot = (
        schedule_results_df.loc[
            (schedule_results_df["Type"] == 1)
            & (schedule_results_df["appliance_name"] == device)
        ].sort_values("block").copy()
    )
    if df_plot.empty:
        raise ValueError(f"Device '{device}' not found among power-flexible devices.")
    if df_plot["block"].duplicated().any():
        raise ValueError("Expected one row per block for the device.")

    blocks = np.arange(int(df_plot["block"].min()), int(df_plot["block"].max()) + 1)
    data = df_plot.set_index("block").reindex(blocks)
    Y_rec = data["Y_rec"].to_numpy(dtype=float)
    Y_FC = data["Y_FC"].to_numpy(dtype=float)
    x = np.append(blocks, blocks[-1] + 1) * block_size
    Y_rec = np.append(Y_rec, Y_rec[-1])
    Y_FC = np.append(Y_FC, Y_FC[-1])

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.step(x, Y_rec, where="post", color="tab:green", linewidth=1.8,
            label=r"$Y^{\mathrm{rec}}$")
    ax.step(x, Y_FC, where="post", color="tab:blue", linestyle="--",
            linewidth=1.5, label=r"$Y^{\mathrm{FC}}$")

    def time_label(minute):
        minute = int(round(minute))
        return f"{minute // 60:02d}:{minute % 60:02d}"

    ticks = np.append(np.arange(x[0], x[-1], tick_step), x[-1])
    ax.set_xticks(ticks)
    ax.set_xticklabels([time_label(t) for t in ticks], rotation=45, ha="right")
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$Y$")
    ax.set_title(f"Power Level — {device} | {time_label(x[0])} - {time_label(x[-1])}")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.tight_layout()
    plt.show()
    return fig, ax


In [ ]:
def plot_stacked_device_power(
    X_matrix,
    Y_matrix,
    Power_max,
    dev_names,
    time_start=0,
    time_finish=None,
    P_lim_vector=None,
    colors=None,
    tick_step=15,
    title="Stacked Device Power"
):
    """
    Plot stacked device-level power for a selected time interval.

    Parameters
    ----------
    X_matrix : ndarray, shape (T, N_DEV)
        ON/OFF state matrix.

    Y_matrix : ndarray, shape (T, N_DEV)
        Power modulation matrix.

    Power_max : array-like, shape (N_DEV,)
        Rated/max power of each device [W].

    dev_names : list of str
        Device names.

    time_start : int
        Start minute of the interval.

    time_finish : int or None
        End minute of the interval.
        If None, uses the full remaining horizon.

    P_lim_vector : array-like, optional
        Minute-level power limit [W].

    colors : list, optional
        Device colors.

    tick_step : int
        X-axis tick spacing in minutes.

    title : str
        Plot title.

    Returns
    -------
    P_dev : ndarray
        Device-level power matrix for the full horizon.

    P_total : ndarray
        Aggregate power vector for the full horizon.
    """

    import numpy as np
    import matplotlib.pyplot as plt

    X_matrix = np.asarray(X_matrix, dtype=float)
    Y_matrix = np.asarray(Y_matrix, dtype=float)
    Power_max = np.asarray(Power_max, dtype=float)

    T, N_DEV = X_matrix.shape

    if time_finish is None:
        time_finish = T

    if not (0 <= time_start < time_finish <= T):
        raise ValueError(
            f"Invalid interval: "
            f"time_start={time_start}, "
            f"time_finish={time_finish}, "
            f"T={T}"
        )

    if Y_matrix.shape != X_matrix.shape:
        raise ValueError(
            "X_matrix and Y_matrix must have the same shape."
        )

    if len(Power_max) != N_DEV:
        raise ValueError(
            "Power_max length must match number of devices."
        )

    if len(dev_names) != N_DEV:
        raise ValueError(
            "dev_names length must match number of devices."
        )

    # ========================================================
    # DEVICE POWER
    # ========================================================

    P_dev = (
        X_matrix
        * Y_matrix
        * Power_max.reshape(1, -1)
    )

    P_total = np.sum(
        P_dev,
        axis=1
    )

    # ========================================================
    # SELECT INTERVAL
    # ========================================================

    P_dev_slice = P_dev[
        time_start:time_finish,
        :
    ]

    minutes = np.arange(
        time_start,
        time_finish
    )

    x = np.arange(
        len(minutes)
    )

    # ========================================================
    # KEEP ONLY DEVICES ACTIVE IN THIS INTERVAL
    # ========================================================

    active_mask = np.any(
        P_dev_slice > 1e-9,
        axis=0
    )

    active_indices = np.where(
        active_mask
    )[0]

    P_dev_active = P_dev_slice[
        :,
        active_indices
    ]

    active_names = [
        dev_names[j]
        for j in active_indices
    ]

    # ========================================================
    # COLORS
    # ========================================================

    active_colors = None

    if colors is not None:

        active_colors = [
            colors[j]
            for j in active_indices
        ]

    # ========================================================
    # PLOT
    # ========================================================

    fig, ax = plt.subplots(
        figsize=(16, 7)
    )

    if len(active_indices) > 0:

        ax.stackplot(
            x,
            P_dev_active.T,
            labels=active_names,
            colors=active_colors,
            alpha=0.9
        )

    # Power limit
    if P_lim_vector is not None:

        P_lim_vector = np.asarray(
            P_lim_vector,
            dtype=float
        )

        P_lim_slice = P_lim_vector[
            time_start:time_finish
        ]

        ax.plot(
            x,
            P_lim_slice,
            linestyle="--",
            linewidth=2,
            label="P_lim"
        )

    # ========================================================
    # TIME LABELS
    # ========================================================

    tick_positions = np.arange(
        0,
        len(minutes),
        tick_step
    )

    tick_minutes = minutes[
        tick_positions
    ]

    tick_labels = [
        f"{m // 60:02d}:{m % 60:02d}"
        for m in tick_minutes
    ]

    ax.set_xticks(
        tick_positions
    )

    ax.set_xticklabels(
        tick_labels,
        rotation=45,
        ha="right"
    )

    # ========================================================
    # FORMATTING
    # ========================================================

    start_label = (
        f"{time_start // 60:02d}:"
        f"{time_start % 60:02d}"
    )

    finish_label = (
        f"{time_finish // 60:02d}:"
        f"{time_finish % 60:02d}"
    )

    ax.set_title(
        f"{title} | "
        f"{start_label} - {finish_label}"
    )

    ax.set_xlabel(
        "Time"
    )

    ax.set_ylabel(
        "Power [W]"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False
    )

    plt.tight_layout()
    plt.show()

    return P_dev, P_total

In [ ]:
def get_p_lim_for_minute(
    minute,
    pv_bess=False
):
    """
    Instantaneous power limit [W] for a given minute.
    """

    tz = get_time_zone(
        minute
    )

    if pv_bess:

        return (
            timezone_power_limits_W_PV_BESS[
                tz
            ]
        )

    return (
        timezone_power_limits_W[
            tz
        ])
    
def get_time_zone(minute):
    """
    Return the time-zone index associated with a minute of the day.

    0 -> 00:00 - 09:59
    1 -> 10:00 - 19:29
    2 -> 19:30 - 22:29
    3 -> 22:30 - 23:59
    """

    minute = int(minute)

    for tz, limits in TIME_ZONES.items():

        if (
            limits["start"]
            <= minute
            < limits["end"]
        ):
            return tz

    raise ValueError(
        f"Minute outside daily horizon: {minute}"
    )
def get_time_zone_from_block(
    k,
    block_size=BLOCK_SIZE
):
    """
    Return time-zone index associated with scheduling block k.
    """

    minute = (
        int(k)
        * block_size
    )

    return get_time_zone(
        minute
    )
def get_e_lim_k(
    k,
    block_size=BLOCK_SIZE
):
    """Return the Day-Ahead energy limit [Wh] for block k."""
    tz = get_time_zone_from_block(
        k,
        block_size
    )
    return float(
        timezone_energy_limits_Wh[tz]
    )


In [ ]:
def compute_weight_boundary_metrics(
    W_matrix,
    Type,
    dev_names,
    W_min=1.0,
    W_max=10.0,
    tol=1e-6
):
    """
    Compute simple boundary metrics for time-flexible devices.

    Metrics:
        - Percentage of time inside the expected weight range
        - Percentage of time above W_max
        - Percentage of time below W_min
        - Maximum observed weight
        - Minimum observed weight
    """

    W_matrix = np.asarray(W_matrix, dtype=float)
    Type = np.asarray(Type)

    rows = []

    for j in np.where(Type == 2)[0]:

        W_j = W_matrix[:, j]

        valid = ~np.isnan(W_j)
        W_valid = W_j[valid]

        if len(W_valid) == 0:
            continue

        below = W_valid < (W_min - tol)
        above = W_valid > (W_max + tol)
        inside = ~(below | above)

        rows.append({
            "Device": dev_names[j],

            "Inside range [%]":
                100.0 * np.mean(inside),

            "Above WT_max [%]":
                100.0 * np.mean(above),

            "Below WT_min [%]":
                100.0 * np.mean(below),

            "Max WT":
                float(np.max(W_valid)),

            "Min WT":
                float(np.min(W_valid))
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_block_energy_stacked(energy_Wh, device_names, energy_limit_Wh,
                              block_size=30, tick_step=120, colors=None,
                              title="Baseline consumption", blocks=None):
    """Stack device energies in Wh, with one bar per scheduling block."""
    import numpy as np
    import matplotlib.pyplot as plt

    energy = np.asarray(energy_Wh, dtype=float)
    limits = np.asarray(energy_limit_Wh, dtype=float)
    if energy.ndim != 2 or energy.shape[1] != len(device_names):
        raise ValueError("Expected one energy column per device.")
    if energy.shape[0] == 0 or limits.shape != (energy.shape[0],):
        raise ValueError("Expected one energy limit per block.")
    if block_size <= 0 or tick_step <= 0 or not np.isfinite(energy).all():
        raise ValueError("Invalid block size, tick step or energy values.")
    blocks = np.arange(len(energy)) if blocks is None else np.asarray(blocks)
    if len(blocks) != len(energy) or not np.all(np.diff(blocks) == 1):
        raise ValueError("Expected consecutive, unique blocks.")
    minutes = blocks * block_size
    edges = np.append(minutes, minutes[-1] + block_size)
    fig, ax = plt.subplots(figsize=(16, 7))
    bottom = np.zeros(len(energy))
    for j, name in enumerate(device_names):
        color = colors[j] if colors is not None else plt.get_cmap("tab20")(j % 20)
        ax.bar(minutes, energy[:, j], bottom=bottom, width=block_size,
               align="edge", color=color, linewidth=0, label=name)
        bottom += energy[:, j]
    ax.stairs(limits, edges, baseline=None, color="black", linestyle="--",
              linewidth=2, label="Block energy limit")
    ticks = np.append(np.arange(edges[0], edges[-1], tick_step), edges[-1])
    ax.set_xticks(ticks, [f"{int(t)//60:02d}:{int(t)%60:02d}" for t in ticks])
    ax.set(xlim=(edges[0], edges[-1]), ylim=(0, None), xlabel="Time of day",
           ylabel=f"Energy per {block_size}-minute block [Wh]", title=title)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_schedule(energy_results_df, block_size=30,
                  title="Day-Ahead Scheduling Results", tick_step=120,
                  colors=None, device_energy_Wh=None, device_names=None):
    """Plot recommended device energy; verify it matches the reported totals."""
    import numpy as np
    summary = energy_results_df.sort_values("block").reset_index(drop=True)
    if device_energy_Wh is None or device_names is None:
        raise ValueError("Provide device_energy_Wh and device_names for the stacked plot.")
    np.testing.assert_allclose(np.asarray(device_energy_Wh).sum(axis=1),
                               summary["E_total_rec_Wh"], rtol=1e-7, atol=1e-5)
    return plot_block_energy_stacked(
        device_energy_Wh, device_names, summary["E_limit_Wh"].to_numpy(),
        block_size=block_size, tick_step=tick_step, colors=colors,
        title=title, blocks=summary["block"].to_numpy(),
    )


In [ ]:
def plot_time_flexible_activation_heatmap(
    schedule_results_df,
    title="Day-Ahead Scheduling Adjustments",
    block_size=15,
    tick_step=120,
    reference_ax=None
):
    """Plot recommended - baseline ON duration per block [min]."""
    title_fontsize = 18
    axis_fontsize = 16
    tick_fontsize = 14
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm

    if block_size <= 0 or tick_step <= 0:
        raise ValueError("block_size and tick_step must be positive.")
    df_plot = schedule_results_df.loc[schedule_results_df["Type"] == 2].copy()
    if df_plot.empty:
        print("No time-flexible devices found.")
        return

    df_plot["delta_n_on"] = df_plot["n_on_rec"] - df_plot["n_on_FC"]
    device_order = df_plot["appliance_name"].drop_duplicates()
    heat = (
        df_plot.pivot(index="appliance_name", columns="block", values="delta_n_on")
        .reindex(device_order).sort_index(axis=1)
    )
    heat = heat.loc[heat.abs().max(axis=1) > 1e-6]
    if heat.empty:
        print("No time-flexible scheduling changes.")
        return

    blocks = np.arange(int(heat.columns.min()), int(heat.columns.max()) + 1)
    heat = heat.reindex(columns=blocks)
    x_edges = np.append(blocks, blocks[-1] + 1) * block_size
    y_edges = np.arange(len(heat) + 1)

    figure_width = 16
    left, right = 0.20, 0.80
    if reference_ax is not None:
        reference_ax.figure.canvas.draw()
        position = reference_ax.get_position()
        figure_width = reference_ax.figure.get_figwidth()
        left, right = position.x0, position.x1

    fig, ax = plt.subplots(
        figsize=(figure_width, max(4, 0.6 * len(heat) + 2)), layout="none"
    )
    fig.subplots_adjust(left=left, right=right, bottom=0.42, top=0.84)
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#eeeeee")
    im = ax.pcolormesh(
        x_edges, y_edges,
        np.ma.masked_invalid(heat.to_numpy(dtype=float)),
        cmap=cmap,
        norm=TwoSlopeNorm(vmin=-block_size, vcenter=0, vmax=block_size),
        shading="flat", edgecolors="none", rasterized=True
    )
    ax.set_yticks(np.arange(len(heat)) + 0.5)
    ax.set_yticklabels(heat.index)
    ax.set_ylabel("")
    ax.invert_yaxis()
    for boundary in range(1, len(heat)):
        ax.axhline(boundary, color="white", linewidth=1)

    def time_label(minute):
        minute = int(round(minute))
        return f"{minute // 60:02d}:{minute % 60:02d}"

    ticks = np.append(np.arange(x_edges[0], x_edges[-1], tick_step), x_edges[-1])
    ax.set_xticks(ticks)
    ax.set_xticklabels([time_label(t) for t in ticks], rotation=45, ha="right")
    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_xlabel("Time", fontsize=axis_fontsize)
    ax.set_title(
        f"{title} | {time_label(x_edges[0])} - {time_label(x_edges[-1])}",
        fontsize=title_fontsize,
    )
    ax.tick_params(
        axis="both", length=0, pad=7, labelsize=tick_fontsize
    )
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    if reference_ax is not None:
        ax.title.set_fontproperties(reference_ax.title.get_fontproperties())
        ax.xaxis.label.set_fontproperties(reference_ax.xaxis.label.get_fontproperties())
        reference_ticks = reference_ax.get_xticklabels()
        if reference_ticks:
            tick_font = reference_ticks[0].get_fontproperties()
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontproperties(tick_font)

    bar_width = 0.55 * (right - left)
    bar_left = (left + right - bar_width) / 2
    cbar_ax = fig.add_axes([bar_left, 0.18, bar_width, 0.025])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")
    color_ticks = np.linspace(-block_size, block_size, 5)
    cbar.set_ticks(color_ticks)
    cbar.set_ticklabels([
        "0" if np.isclose(value, 0) else f"{value:+g}"
        for value in color_ticks
    ])
    cbar.set_label(
        r"$\Delta n_{\mathrm{ON}}$ [min]",
        fontsize=axis_fontsize,
    )
    cbar.ax.xaxis.label.set_fontproperties(ax.xaxis.label.get_fontproperties())
    tick_font = ax.get_xticklabels()[0].get_fontproperties()
    for label in cbar.ax.get_xticklabels():
        label.set_fontproperties(tick_font)
    cbar.ax.tick_params(length=0, pad=5, labelsize=tick_fontsize)
    cbar.outline.set_visible(False)
    plt.show()
    return fig, ax


In [ ]:
def plot_device_weight_and_activation(
    device,
    W_matrix,
    X_matrix,
    dev_names,
    Type=None,
    timezone_limits=None,
    timezone_labels=None,
    figsize=(16, 5)
):
    """
    Plot the time-weight trajectory and ON/OFF activation
    of a single device.

    The two vertical axes are aligned so that:

        OFF = 0  aligns with W_T = 1
        ON  = 1  aligns with W_T = 10

    Parameters
    ----------
    device : str or int
        Device name or index.

    W_matrix : ndarray, shape (T, N_DEV)
        Matrix containing time weights.

    X_matrix : ndarray, shape (T, N_DEV)
        Binary ON/OFF state matrix.

    dev_names : list of str
        Device names.

    Type : array-like, optional
        Device type vector.

    timezone_limits : dict, optional
        Mapping:

            timezone -> (start, end)

        Example:

            {
                0: (0, 360),
                1: (360, 720),
                2: (720, 1080),
                3: (1080, 1440)
            }

    timezone_labels : dict, optional
        Labels to display for each timezone.

        Example:

            {
                0: "Night",
                1: "Morning",
                2: "Afternoon",
                3: "Evening"
            }

    figsize : tuple, default=(16, 5)
        Figure size.
    """

    W_matrix = np.asarray(W_matrix, dtype=float)
    X_matrix = np.asarray(X_matrix)

    if W_matrix.ndim != 2:
        raise ValueError(
            "W_matrix must have shape (T, N_DEV)."
        )

    if X_matrix.ndim != 2:
        raise ValueError(
            "X_matrix must have shape (T, N_DEV)."
        )

    if W_matrix.shape != X_matrix.shape:
        raise ValueError(
            "W_matrix and X_matrix must have the same shape."
        )

    if len(dev_names) != W_matrix.shape[1]:
        raise ValueError(
            "dev_names must contain one name per device."
        )

    # --------------------------------------------------
    # Select device
    # --------------------------------------------------
    if isinstance(device, str):

        exact_matches = [
            j
            for j, name in enumerate(dev_names)
            if name.lower() == device.lower()
        ]

        if exact_matches:
            j = exact_matches[0]

        else:
            partial_matches = [
                j
                for j, name in enumerate(dev_names)
                if device.lower() in name.lower()
            ]

            if len(partial_matches) == 0:
                raise ValueError(
                    f"Device '{device}' not found."
                )

            if len(partial_matches) > 1:
                matching_names = [
                    dev_names[index]
                    for index in partial_matches
                ]

                raise ValueError(
                    f"Several devices match '{device}': "
                    f"{matching_names}"
                )

            j = partial_matches[0]

    elif isinstance(device, (int, np.integer)):

        j = int(device)

        if j < 0 or j >= len(dev_names):
            raise IndexError(
                f"Device index must be between 0 and "
                f"{len(dev_names) - 1}."
            )

    else:
        raise TypeError(
            "'device' must be a device name or integer index."
        )

    if Type is not None:

        Type = np.asarray(Type)

        if Type[j] != 2:
            raise ValueError(
                f"{dev_names[j]} is not time-flexible "
                f"(Type = {Type[j]})."
            )

    T = W_matrix.shape[0]
    time = np.arange(T)

    fig, ax_weight = plt.subplots(
        figsize=figsize
    )

    # --------------------------------------------------
    # Timezone background
    # --------------------------------------------------
    timezone_handles = []

    if timezone_limits is not None:

        timezone_colors = [
            "tab:blue",
            "tab:orange",
            "tab:green",
            "tab:purple",
            "tab:brown",
            "tab:pink",
        ]

        for position, (timezone, limits) in enumerate(
            timezone_limits.items()
        ):

            start, end = limits

            start = max(0, int(start))
            end = min(T, int(end))

            color = timezone_colors[
                position % len(timezone_colors)
            ]

            ax_weight.axvspan(
                start,
                end,
                color=color,
                alpha=0.08,
                linewidth=0,
                zorder=0
            )

            # Boundary at the beginning of each timezone,
            # except at t = 0.
            if start > 0:
                ax_weight.axvline(
                    start,
                    linestyle=":",
                    linewidth=1.2,
                    color=color,
                    alpha=0.7,
                    zorder=1
                )

            if timezone_labels is None:
                label = f"Timezone {timezone}"
            else:
                label = timezone_labels.get(
                    timezone,
                    f"Timezone {timezone}"
                )

            timezone_handles.append(
                Patch(
                    facecolor=color,
                    alpha=0.15,
                    label=label
                )
            )

            # Label at the center of the timezone.
            midpoint = start + (end - start) / 2

            ax_weight.text(
                midpoint,
                10.25,
                label,
                horizontalalignment="center",
                verticalalignment="center",
                fontsize=9,
                alpha=0.75
            )

    # --------------------------------------------------
    # Weight trajectory
    # --------------------------------------------------
    weight_line, = ax_weight.plot(
        time,
        W_matrix[:, j],
        linewidth=2,
        label=r"$W_T$",
        zorder=4
    )

    min_line = ax_weight.axhline(
        1,
        linestyle="--",
        linewidth=1,
        alpha=0.6,
        label=r"$W_T=1$",
        zorder=2
    )

    max_line = ax_weight.axhline(
        10,
        linestyle="--",
        linewidth=1,
        alpha=0.6,
        label=r"$W_T=10$",
        zorder=2
    )

    weight_ymin = 0.5
    weight_ymax = 10.5

    ax_weight.set_ylim(
        weight_ymin,
        weight_ymax
    )

    ax_weight.set_xlim(
        0,
        max(T - 1, 1)
    )

    ax_weight.set_xlabel(
        "Minute of day"
    )

    ax_weight.set_ylabel(
        r"Time weight $W_T$"
    )

    ax_weight.set_yticks(
        np.arange(1, 11)
    )

    ax_weight.grid(
        axis="both",
        alpha=0.25,
        zorder=1
    )

    # --------------------------------------------------
    # ON/OFF activation axis
    # --------------------------------------------------
    ax_state = ax_weight.twinx()

    activation_bars = ax_state.bar(
        time,
        X_matrix[:, j],
        width=1.0,
        alpha=0.20,
        label="ON/OFF state",
        zorder=3
    )

    # --------------------------------------------------
    # Align OFF=0 with W=1 and ON=1 with W=10
    # --------------------------------------------------
    weight_range = (
        weight_ymax - weight_ymin
    )

    off_position = (
        (1.0 - weight_ymin)
        / weight_range
    )

    on_position = (
        (10.0 - weight_ymin)
        / weight_range
    )

    state_range = (
        1.0 / (on_position - off_position)
    )

    state_ymin = (
        -off_position * state_range
    )

    state_ymax = (
        state_ymin + state_range
    )

    ax_state.set_ylim(
        state_ymin,
        state_ymax
    )

    ax_state.set_yticks([0, 1])

    ax_state.set_yticklabels([
        "OFF",
        "ON"
    ])

    ax_state.set_ylabel(
        "Appliance state"
    )

    ax_state.grid(False)

    # --------------------------------------------------
    # Title and legend
    # --------------------------------------------------
    ax_weight.set_title(
        f"Time weight and activation — {dev_names[j]}"
    )

    handles_weight, labels_weight = (
        ax_weight.get_legend_handles_labels()
    )

    handles_state, labels_state = (
        ax_state.get_legend_handles_labels()
    )

    handles = (
        handles_weight
        + handles_state
        + timezone_handles
    )

    labels = (
        labels_weight
        + labels_state
        + [
            handle.get_label()
            for handle in timezone_handles
        ]
    )

    ax_weight.legend(
        handles,
        labels,
        loc="upper right",
        ncol=2
    )

    plt.tight_layout()
    plt.show()

# Data preprocessing

## PV Production

In [ ]:
day_start="2026-07-17 00:00"
T=24*60
installed_power_kWp_pv=40.0
inverter_power_max_kW_pv=35.0
pv_forecast_kW = np.zeros(T)

# 08:00–10:00: ramp-up
for minute in range(8 * 60, 10 * 60):
    pv_forecast_kW[minute] = (
        (minute - 8 * 60)
        / (2 * 60)
        * 6.0
    )

# 10:00–13:00: stronger production
for minute in range(10 * 60, 13 * 60):
    pv_forecast_kW[minute] = (
        6.0
        + (minute - 10 * 60)
        / (3 * 60)
        * 6.0
    )

# 13:00–15:00: around peak
for minute in range(13 * 60, 15 * 60):
    pv_forecast_kW[minute] = 12.0

# 15:00–18:00: ramp-down
for minute in range(15 * 60, 18 * 60):
    pv_forecast_kW[minute] = (
        12.0
        * (
            1.0
            - (minute - 15 * 60)
            / (3 * 60)
        )
    )
def characterize_pv_numpy(
    pv_forecast_kW,
    day_start="2026-01-01 00:00",
    block_size_min=15,
    installed_power_kWp=None,
    inverter_power_max_kW=None
):
    """
    Characterizes a minute-level PV forecast provided as a NumPy array.

    Parameters
    ----------
    pv_forecast_kW : numpy.ndarray
        Minute-resolution PV forecast [kW].
        For one complete day, it should contain 1440 values.

    day_start : str or pandas.Timestamp
        Timestamp corresponding to the first value of the forecast.

    block_size_min : int
        Optimization block duration [minutes].

    installed_power_kWp : float, optional
        Installed PV capacity [kWp].

    inverter_power_max_kW : float, optional
        Maximum inverter output power [kW].

    Returns
    -------
    pv_characterization : pandas.DataFrame
        One row per optimization block.

    pv_info : dict
        Global PV parameters.
    """

    # Convert input to a 1D NumPy array
    pv_forecast_kW = np.asarray(
        pv_forecast_kW,
        dtype=float
    ).reshape(-1)

    if len(pv_forecast_kW) == 0:
        raise ValueError("pv_forecast_kW cannot be empty.")

    if block_size_min <= 0:
        raise ValueError("block_size_min must be greater than zero.")

    # Replace invalid values and prevent negative PV generation
    pv_forecast_raw_kW = np.nan_to_num(
        pv_forecast_kW,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    pv_forecast_raw_kW = np.clip(
        pv_forecast_raw_kW,
        a_min=0.0,
        a_max=None
    )

    # -------------------------
    # Physical power limit
    # -------------------------
    power_limits = []

    if installed_power_kWp is not None:
        power_limits.append(float(installed_power_kWp))

    if inverter_power_max_kW is not None:
        power_limits.append(float(inverter_power_max_kW))

    if power_limits:
        effective_power_limit_kW = min(power_limits)

        pv_available_kW = np.minimum(
            pv_forecast_raw_kW,
            effective_power_limit_kW
        )
    else:
        effective_power_limit_kW = None
        pv_available_kW = pv_forecast_raw_kW.copy()

    # -------------------------
    # Timestamp vector
    # -------------------------
    timestamps = pd.date_range(
        start=pd.Timestamp(day_start),
        periods=len(pv_forecast_raw_kW),
        freq="min"
    )

    # -------------------------
    # Block assignment
    # -------------------------
    block = np.arange(len(pv_forecast_raw_kW)) // block_size_min

    minute_data = pd.DataFrame({
        "timestamp": timestamps,
        "block": block,
        "PV_forecast_raw_kW": pv_forecast_raw_kW,
        "PV_available_kW": pv_available_kW
    })

    minute_data["PV_forecast_kWh_min"] = (
        minute_data["PV_forecast_raw_kW"] / 60.0
    )

    minute_data["PV_available_kWh_min"] = (
        minute_data["PV_available_kW"] / 60.0
    )

    # -------------------------
    # Aggregate blocks
    # -------------------------
    pv_characterization = (
        minute_data
        .groupby("block", as_index=False)
        .agg(
            block_start=("timestamp", "first"),

            PV_forecast_kW_mean=(
                "PV_forecast_raw_kW",
                "mean"
            ),

            PV_available_kW_mean=(
                "PV_available_kW",
                "mean"
            ),

            PV_forecast_kWh=(
                "PV_forecast_kWh_min",
                "sum"
            ),

            PV_available_kWh=(
                "PV_available_kWh_min",
                "sum"
            ),

            observed_minutes=(
                "timestamp",
                "count"
            )
        )
    )

    pv_characterization["PV_clipped_kWh"] = (
        pv_characterization["PV_forecast_kWh"]
        - pv_characterization["PV_available_kWh"]
    )

    pv_characterization["complete_block"] = (
        pv_characterization["observed_minutes"]
        == block_size_min
    )

    pv_characterization = pv_characterization[
        [
            "block",
            "block_start",
            "PV_forecast_kW_mean",
            "PV_available_kW_mean",
            "PV_forecast_kWh",
            "PV_available_kWh",
            "PV_clipped_kWh",
            "complete_block"
        ]
    ]

    # -------------------------
    # Global information
    # -------------------------
    pv_info = {
        "installed_power_kWp": installed_power_kWp,
        "inverter_power_max_kW": inverter_power_max_kW,
        "effective_power_limit_kW": effective_power_limit_kW,
        "block_size_min": block_size_min,
        "number_of_minutes": len(pv_forecast_raw_kW),
        "number_of_blocks": len(pv_characterization)
    }

    return pv_characterization, pv_info

pv_characterization, pv_info = characterize_pv_numpy(
    pv_forecast_kW=pv_forecast_kW,
    day_start=day_start,
    block_size_min=15,
    installed_power_kWp=installed_power_kWp_pv,
    inverter_power_max_kW=inverter_power_max_kW_pv
)
display(
    pv_characterization[
        pv_characterization["PV_available_kW_mean"] > 0
    ]
)

## Battery Info

In [ ]:
def characterize_battery(
    blocks,
    capacity_kWh,
    soc_init_kWh,
    soc_min_kWh,
    soc_max_kWh,
    charge_power_max_kW,
    discharge_power_max_kW,
    eta_charge,
    eta_discharge,
    block_size_min=15
):
    """
    Creates the physical characterization of a battery for each
    optimization block.

    Parameters
    ----------
    blocks : iterable
        Optimization block identifiers.

    capacity_kWh : float
        Nominal or usable battery capacity [kWh].

    soc_init_kWh : float
        Initial stored energy at the beginning of the horizon [kWh].

    soc_min_kWh : float
        Minimum allowed stored energy [kWh].

    soc_max_kWh : float
        Maximum allowed stored energy [kWh].

    charge_power_max_kW : float
        Maximum battery charging power [kW].

    discharge_power_max_kW : float
        Maximum battery discharging power [kW].

    eta_charge : float
        Charging efficiency, between 0 and 1.

    eta_discharge : float
        Discharging efficiency, between 0 and 1.

    block_size_min : int or float
        Duration of each optimization block [minutes].

    Returns
    -------
    battery_characterization : pandas.DataFrame
        Battery physical parameters for each optimization block.

    battery_info : dict
        Global battery parameters.
    """

    # -------------------------
    # Input validation
    # -------------------------
    if capacity_kWh <= 0:
        raise ValueError("capacity_kWh must be greater than zero.")

    if not 0 <= soc_min_kWh <= soc_init_kWh <= soc_max_kWh:
        raise ValueError(
            "The SoC values must satisfy: "
            "soc_min_kWh <= soc_init_kWh <= soc_max_kWh."
        )

    if soc_max_kWh > capacity_kWh:
        raise ValueError(
            "soc_max_kWh cannot be greater than capacity_kWh."
        )

    if charge_power_max_kW < 0:
        raise ValueError(
            "charge_power_max_kW cannot be negative."
        )

    if discharge_power_max_kW < 0:
        raise ValueError(
            "discharge_power_max_kW cannot be negative."
        )

    if not 0 < eta_charge <= 1:
        raise ValueError(
            "eta_charge must be between 0 and 1."
        )

    if not 0 < eta_discharge <= 1:
        raise ValueError(
            "eta_discharge must be between 0 and 1."
        )

    if block_size_min <= 0:
        raise ValueError(
            "block_size_min must be greater than zero."
        )

    # -------------------------
    # Time conversion
    # -------------------------
    block_duration_h = block_size_min / 60.0

    # Maximum grid-side energy exchanged during one block
    charge_energy_max_kWh = (
        charge_power_max_kW
        * block_duration_h
    )

    discharge_energy_max_kWh = (
        discharge_power_max_kW
        * block_duration_h
    )

    # -------------------------
    # SoC percentages
    # -------------------------
    soc_init_pct = (
        100
        * soc_init_kWh
        / capacity_kWh
    )

    soc_min_pct = (
        100
        * soc_min_kWh
        / capacity_kWh
    )

    soc_max_pct = (
        100
        * soc_max_kWh
        / capacity_kWh
    )

    # -------------------------
    # Per-block characterization
    # -------------------------
    battery_characterization = pd.DataFrame({
        "block": list(blocks),
        "block_size_min": block_size_min,
        "block_duration_h": block_duration_h,
        "capacity_kWh": capacity_kWh,
        "soc_min_kWh": soc_min_kWh,
        "soc_max_kWh": soc_max_kWh,
        "charge_power_max_kW": charge_power_max_kW,
        "discharge_power_max_kW": discharge_power_max_kW,
        "charge_energy_max_kWh": charge_energy_max_kWh,
        "discharge_energy_max_kWh": discharge_energy_max_kWh,
        "eta_charge": eta_charge,
        "eta_discharge": eta_discharge
    })

    # -------------------------
    # Global information
    # -------------------------
    battery_info = {
        "capacity_kWh": capacity_kWh,
        "soc_init_kWh": soc_init_kWh,
        "soc_min_kWh": soc_min_kWh,
        "soc_max_kWh": soc_max_kWh,
        "soc_init_pct": soc_init_pct,
        "soc_min_pct": soc_min_pct,
        "soc_max_pct": soc_max_pct,
        "charge_power_max_kW": charge_power_max_kW,
        "discharge_power_max_kW": discharge_power_max_kW,
        "charge_energy_max_kWh": charge_energy_max_kWh,
        "discharge_energy_max_kWh": discharge_energy_max_kWh,
        "eta_charge": eta_charge,
        "eta_discharge": eta_discharge,
        "block_size_min": block_size_min,
        "block_duration_h": block_duration_h
    }

    return battery_characterization, battery_info

battery_characterization, battery_info = characterize_battery(
    blocks=range(N_BLOCKS),
    capacity_kWh=20.0,
    soc_init_kWh=10.0,
    soc_min_kWh=2.0,
    soc_max_kWh=18.0,
    charge_power_max_kW=5.0,
    discharge_power_max_kW=5.0,
    eta_charge=0.95,
    eta_discharge=0.95,
    block_size_min=BLOCK_SIZE
)

battery_characterization.head()

## Consumption Data

In [ ]:
Year=2019
Month=2
Day=14
N_DEV = 13

# instante inicial e final do dia em análise
day_start_dt = datetime(Year, Month, Day, 0, 0, 0)
day_end_dt = day_start_dt + timedelta(days=1)

# =========================
# DEVICE GROUPS
# =========================
NON_OPERATIONAL_DEVICES = {0, 1, 2, 3, 4}
OPERATIONAL_DEVICES = {5, 6}
MISSING_STATE_DURATION = 1e-6
NON_FLEXIBLE_DEVICES = {7, 8, 9}  # Glass Washer, Dish Washer, Convection Oven
POWER_FLEXIBLE_DEVICES = {10, 11, 12}  # Light1, Light2, Exhaust
EXTERNAL_PROFILE_DEVICES = {12}  # Exhaust

POWER_FLEXIBLE_SCHEDULES = {
    10: {
        "name": "Light1",
        "start": 10 * 60,          # 10:00
        "end": 22 * 60 + 30        # 22:30
    },
    11: {
        "name": "Light2",
        "start": 10 * 60,          # 10:00
        "end": 22 * 60 + 30        # 22:30
    }
}
EXTERNAL_PROFILE_DATES = {
    12: "2019-03-11"
}

external_activation_files = {
    12: "05_Exhaust_activations.xlsx"
}

N_MIN_ACTIVATIONS = 2
T_NO_ACTIVITY_FALLBACK = 5000
T_MIN = 5
T_OFF_FORCED_ON = 5
DAY_MINUTES = 24 * 60
T= DAY_MINUTES

# diretório dos ficheiros
diretorio = '../Data'

# mapeamento appliance -> ficheiro de ativações
activation_files = {
    0: "activations_Drinks_Refrigerator.csv",
    1: "activations_Refrigerator_2.csv",
    2: "activations_Refrigerator_3.csv",
    3: "activations_Refrigerator_4.csv",
    4: "activations_Garde_Manger_10.csv",
    5: "activations_Dual_Fryer.csv",
    6: "activations_Salamander_1.csv",
    7: "activations_Glass_Washer.csv",
    8: "activations_Dish_Washer.csv",
    9: "activations_Convection_Oven_1.csv",
}

DEVICE_NAMES = {
    0: "Drinks Refrigerator",
    1: "Refrigerator 2",
    2: "Refrigerator 3",
    3: "Refrigerator 4",
    4: "Garde Manger",
    5: "Dual Fryer",
    6: "Salamander",
    7: "Glass Washer",
    8: "Dish Washer",
    9: "Convection Oven",
    10: "Light1",
    11: "Light2",
    12: "Exhaust",
}

colors = [
    "#4F81BD",  # Drinks Refrigerator
    "#C0504D",  # Blast Chiller
    "#9BBB59",  # Garde Manger
    "#8064A2",  # Ice Machine
    "#4BACC6",  # Dual Fryer
    "#F79646",  # Salamander
    "#385D8A",  # Infrared Shelf
    "#943634",  # Glass Washer
    "#76923C",  # Dish Washer
    "#5F497A",  # Convection Oven
    "#FFD700",  # Light1
    "#EEFF00",  # Light2
    "#708090",  # Exhaust
    
]

In [ ]:
# ======================================
# LEITURA DO INFO.XLSX
# ======================================

df_info = pd.read_excel('../Data/Info.xlsx')

print("Colunas do Info.xlsx:")
print(df_info.columns.tolist())

# se quiseres confirmar o número de equipamentos do Excel
print("\nNúmero de linhas no Info.xlsx:", len(df_info))

# opcional: garantir consistência
if len(df_info) != N_DEV:
    raise ValueError(f"N_DEV={N_DEV}, mas o Info.xlsx tem {len(df_info)} linhas.")

# parâmetros por equipamento
dev_names = df_info["Dev"].tolist()

# índice do tipo de flexibilidade
Type = df_info["Type"].to_numpy(dtype=int)

# estado inicial
# para já vamos ler do Excel; mais tarde podes forçar a zero ou atualizar dinamicamente
State_Init = df_info["State_Init"].to_numpy(dtype=int)

# potência máxima
Power_max = df_info["Power_max"].to_numpy(dtype=float)

# pesos da componente de potência
Weight_P_max = df_info["Weight_P_max"].to_numpy(dtype=float)
Weight_P_min = df_info["Weight_P_min"].to_numpy(dtype=float)

# percentagens de potência
# MUITO IMPORTANTE: converter de % para fração
Y_max = df_info["Y_max (%)"].to_numpy(dtype=float) / 100.0
Y_min = df_info["Y_min (%)"].to_numpy(dtype=float) / 100.0

# tempo mínimo para manter alteração de potência
TimeHoldPower = df_info["TimeHoldPower"].to_numpy(dtype=float)

# impressão de controlo
print("\nResumo dos equipamentos:")
for j in range(N_DEV):
    print(
        f"{j}: {dev_names[j]} | "
        f"Type={Type[j]} | "
        f"State_Init={State_Init[j]} | "
        f"Pmax={Power_max[j]} | "
        f"Ymin={Y_min[j]} | "
        f"Ymax={Y_max[j]}"
        f" | TimeHoldPower={TimeHoldPower[j]}"
    )

In [ ]:
def minute_of_day(ts):
    m = ts.hour * 60 + ts.minute
    if ts.second >= 30:
        m += 1
    return min(m, 1439)

def load_activation_day_csv(diretorio, nome_do_arquivo, target_date):
    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)
    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    # Os CSVs usam separadores e capitalização diferentes.
    df = pd.read_csv(
        caminho_arquivo,
        sep=None,
        engine="python"
    ).iloc[:, 0:3].copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )
    df["first_min_on"] = pd.to_datetime(df["first_min_on"], utc=True, errors="coerce")
    df["last_min_on"]  = pd.to_datetime(df["last_min_on"],  utc=True, errors="coerce")

    df["first_min_on"] = df["first_min_on"].dt.tz_convert(None)
    df["last_min_on"]  = df["last_min_on"].dt.tz_convert(None)

    df = df.dropna(subset=["first_min_on", "last_min_on"]).sort_values("first_min_on").reset_index(drop=True)

    target_date = pd.to_datetime(target_date).date()
    df_day = df[df["first_min_on"].dt.date == target_date].copy().reset_index(drop=True)

    return df_day
def build_X_day_for_device(j, nome_do_arquivo, diretorio, target_date, X_day):
    """
    Constrói X_day[:, j] para o dia inteiro.
    """
    X_day[:, j] = 0

    df_day = load_activation_day_csv(diretorio, nome_do_arquivo, target_date)

    if df_day.empty:
        return

    for _, row in df_day.iterrows():
        start = minute_of_day(row["first_min_on"])
        end   = minute_of_day(row["last_min_on"])

        if end >= start:
            X_day[start:end + 1, j] = 1
        else:
            X_day[start:1440, j] = 1
            X_day[0:end + 1, j] = 1
def build_external_profile_from_activation_file(
    diretorio,
    nome_do_arquivo,
    target_date,
    T=1440
):

    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)

    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    df = pd.read_excel(caminho_arquivo).iloc[:, 0:3].copy()

    df.columns = df.columns.str.strip()

    df["First_min_on"] = pd.to_datetime(
        df["First_min_on"],
        utc=True,
        errors="coerce"
    )

    df["Last_min_on"] = pd.to_datetime(
        df["Last_min_on"],
        utc=True,
        errors="coerce"
    )

    df["First_min_on"] = df["First_min_on"].dt.tz_convert(None)
    df["Last_min_on"] = df["Last_min_on"].dt.tz_convert(None)

    # IMPORTANTÍSSIMO
    target_date = pd.to_datetime(target_date).date()

    df = df[
        df["First_min_on"].dt.date == target_date
    ].copy()

    x = np.zeros(T, dtype=int)

    for _, row in df.iterrows():

        start = minute_of_day(row["First_min_on"])
        end = minute_of_day(row["Last_min_on"])

        if end >= start:
            x[start:end + 1] = 1
        else:
            x[start:T] = 1
            x[0:end + 1] = 1

    return x

def build_power_flexible_schedule(T, start_minute, end_minute):
    x = np.zeros(T, dtype=int)

    if end_minute > start_minute:
        x[start_minute:end_minute] = 1
    else:
        # caso atravesse meia-noite
        x[start_minute:T] = 1
        x[0:end_minute] = 1

    return x

In [ ]:
# 1) Construir X_day global apenas para devices com ficheiro
X_day = np.zeros((T, N_DEV), dtype=int)

for j, nome_do_arquivo in activation_files.items():
    build_X_day_for_device(
        j=j,
        nome_do_arquivo=nome_do_arquivo,
        diretorio=diretorio,
        target_date=day_start_dt.date(),
        X_day=X_day
    )

# 2) Inicializar matrizes
X_onoff_FC_matrix = np.zeros((T, N_DEV), dtype=int)

target_date = day_start_dt.date()

# 3) Preencher baseline com ativações reais
X_onoff_FC_matrix[:, :] = X_day[:T, :]

print("\nX_onoff_FC_matrix (baseline) for the day:")
print(X_onoff_FC_matrix)

In [ ]:
# 1) Adicionar devices power-flexible simulados
for j in POWER_FLEXIBLE_DEVICES:

    # só os que têm horário definido manualmente
    if j not in POWER_FLEXIBLE_SCHEDULES:
        continue

    sched = POWER_FLEXIBLE_SCHEDULES[j]

    X_onoff_FC_matrix[:, j] = build_power_flexible_schedule(
        T=T,
        start_minute=sched["start"],
        end_minute=sched["end"]
    )
# 2) Adicionar perfis externos
for j in EXTERNAL_PROFILE_DEVICES:

    X_onoff_FC_matrix[:, j] = build_external_profile_from_activation_file(
        diretorio=diretorio,
        nome_do_arquivo=external_activation_files[j],
        target_date=EXTERNAL_PROFILE_DATES[j],
        T=T
    )
    
# =========================
# BUILD Y_FC_MATRIX
# =========================

Y_FC_matrix = np.zeros((T, N_DEV), dtype=float)

for j in range(N_DEV):

    if Type[j] == 1:  # power-flexible
        Y_FC_matrix[:, j] = X_onoff_FC_matrix[:, j] * Y_max[j]

    else:
        # Para devices não power-flexible, Y não é relevante.
        # Mas pode ficar igual a X para manter consistência.
        Y_FC_matrix[:, j] = X_onoff_FC_matrix[:, j]

In [ ]:
# ============================================================
# SHARED FIXED SERVICE LIMITS — DAY-AHEAD AND REAL-TIME
#
# P_lim is fixed independently for each timezone.
# Day-Ahead converts the same P_lim into a block energy budget:
#
#     E_lim = P_lim * BLOCK_SIZE / 60
#
# The integrated RT controller reuses P_lim without conversion.
# ============================================================

baseline_power_W = (
    X_onoff_FC_matrix
    * Y_FC_matrix
    * Power_max[np.newaxis, :]
).sum(axis=1)

# Baseline peaks are diagnostic only; they do not define the limits.
timezone_baseline_peaks_W = {
    tz: float(
        np.max(
            baseline_power_W[
                limits["start"]:limits["end"]
            ]
        )
    )
    for tz, limits in TIME_ZONES.items()
}

timezone_power_limits_W = TIMEZONE_POWER_LIMITS_W.copy()

timezone_energy_limits_Wh = {
    tz: P_lim_W * BLOCK_SIZE / 60.0
    for tz, P_lim_W in timezone_power_limits_W.items()
}

P_lim_reference_vector = np.array(
    [get_p_lim_for_minute(minute) for minute in range(T)],
    dtype=float,
)

E_lim_reference_vector = np.array(
    [
        timezone_energy_limits_Wh[
            get_time_zone_from_block(k)
        ]
        for k in range(N_BLOCKS)
    ],
    dtype=float,
)

# Validate the DA/RT conversion for every block.
for k in range(N_BLOCKS):
    start = k * BLOCK_SIZE
    expected_E_lim_Wh = (
        P_lim_reference_vector[start]
        * BLOCK_SIZE
        / 60.0
    )
    if not np.isclose(
        E_lim_reference_vector[k],
        expected_E_lim_Wh,
    ):
        raise RuntimeError(
            f"Inconsistent DA/RT limit in block {k}: "
            f"E_lim={E_lim_reference_vector[k]:.6f} Wh, "
            f"expected={expected_E_lim_Wh:.6f} Wh."
        )

limit_setup = pd.DataFrame({
    "Timezone": list(TIME_ZONES),
    "Baseline peak [kW]": [
        timezone_baseline_peaks_W[tz] / 1000.0
        for tz in TIME_ZONES
    ],
    "P_lim [kW]": [
        timezone_power_limits_W[tz] / 1000.0
        for tz in TIME_ZONES
    ],
    "E_lim per block [Wh]": [
        timezone_energy_limits_Wh[tz]
        for tz in TIME_ZONES
    ],
})

display(
    limit_setup.style.hide(axis="index").format(precision=3)
)


# Device characterization

## Device characterization

In [ ]:
@dataclass
class EpisodeT:
    """
    OFF-ON appliance episode.

    For complete episodes, both the OFF and ON blocks are observed.

    With nearest, only complete OFF -> ON episodes are returned,
    except for single-state operational windows with an epsilon duration.
    The observed/one policies also return completed boundary segments.

    start
        First observed index belonging to the episode.

    end
        Last observed index belonging to the episode.

    T_OFF
        Observed or estimated OFF duration.

    T_ON
        Observed or estimated ON duration.

    timezone
        Timezone associated with the episode.
    """

    start: int
    end: int

    T_OFF: float
    T_ON: float

    timezone: int


def get_operation_window(X):
    """Return first-to-last ON sample as a slice (stop is exclusive).

    No ON samples means an empty window. Internal OFF blocks are retained.
    """
    X = np.asarray(X)
    if X.ndim != 1 or not np.all(np.isin(X, [0, 1])):
        raise ValueError("X must be a one-dimensional binary profile.")
    active = np.flatnonzero(X == 1)
    if active.size == 0:
        return slice(0, 0)
    return slice(int(active[0]), int(active[-1]) + 1)


def extract_off_on_episodes(
    X,
    incomplete_policy="observed",
    minimum_duration=1, 
    index_offset=0,
    timezone=0,
    operational=False,
    missing_duration=1e-6
):
    """
    Extract OFF-ON episodes and complete boundary episodes according
    to an incomplete-episode policy.

    Complete episode
    ----------------
    A complete episode consists of:

        OFF block -> ON block

    Its durations are assigned directly:

        T_OFF = observed OFF-block duration
        T_ON  = observed ON-block duration

    Incomplete episodes
    -------------------
    Initial ON block:
        The preceding OFF block is not observed.

    Final OFF block:
        The following ON block is not observed.

    Policies
    --------
    observed:
        The missing duration is set equal to the duration of the
        observed block.

    nearest:
        Only complete OFF -> ON episodes are returned. Boundary samples
        inherit BOTH durations from the nearest complete episode in
        compute_ton_toff. Boundary stretches are not counted as episodes.

    one:
        The missing duration is set to minimum_duration.

    Parameters
    ----------
    X : array-like, shape (T,)
        Binary appliance-state vector.

    index_offset : int, default=0
        Offset used to convert local indices into global indices.

    incomplete_policy : {"observed", "nearest", "one"},
        default="observed"
        Policy used to complete boundary episodes.

    minimum_duration : int, default=1
        Minimum accepted duration for ordinary episodes.

    operational : bool, default=False
        Restrict episodes to the first-to-last ON window. If this window
        has only one state, return one episode with missing_duration for
        the absent state, including when the policy is nearest.

    missing_duration : float, default=1e-6
        Positive numerical substitute for a state absent from the entire
        operational window. Does not replace durations in ordinary cycles.

    Returns
    -------
    episodes : list of EpisodeT
        Complete episodes for nearest; complete episodes and completed
        boundary segments for observed/one, ordered in time.
    """

    X = np.asarray(X, dtype=int)

    if X.ndim != 1:
        raise ValueError(
            "X must be one-dimensional."
        )

    if not np.all(np.isin(X, [0, 1])):
        raise ValueError(
            "X must contain only 0 and 1."
        )

    valid_policies = {
        "observed",
        "nearest",
        "one",
    }

    if incomplete_policy not in valid_policies:
        raise ValueError(
            "incomplete_policy must be "
            "'observed', 'nearest' or 'one'."
        )

    if minimum_duration < 1:
        raise ValueError(
            "minimum_duration must be at least 1."
        )

    if not np.isfinite(missing_duration) or missing_duration <= 0:
        raise ValueError("missing_duration must be finite and positive.")

    if operational:
        window = get_operation_window(X)
        X = X[window]
        index_offset += window.start

    T = len(X)

    if T == 0:
        return []

    # An all-ON operational window has no complete OFF -> ON episode.
    # Preserve its observed duration; the absent state receives epsilon.
    if operational and np.all(X == X[0]):
        return [EpisodeT(
            start=index_offset,
            end=index_offset + T - 1,
            T_OFF=float(T) if X[0] == 0 else float(missing_duration),
            T_ON=float(T) if X[0] == 1 else float(missing_duration),
            timezone=timezone,
        )]

    # --------------------------------------------------
    # 1. Extract consecutive ON/OFF blocks
    # --------------------------------------------------
    runs = []

    run_start = 0
    current_state = X[0]

    for t in range(1, T):

        if X[t] != current_state:

            run_end = t - 1

            runs.append({
                "state": int(current_state),
                "start": int(run_start),
                "end": int(run_end),
                "duration": int(
                    run_end - run_start + 1
                ),
            })

            run_start = t
            current_state = X[t]

    # Close final block.
    run_end = T - 1

    runs.append({
        "state": int(current_state),
        "start": int(run_start),
        "end": int(run_end),
        "duration": int(
            run_end - run_start + 1
        ),
    })

    # --------------------------------------------------
    # 2. Identify complete OFF -> ON episodes
    # --------------------------------------------------
    complete_episodes = []

    for run_index in range(len(runs) - 1):

        off_run = runs[run_index]
        on_run = runs[run_index + 1]

        if (
            off_run["state"] == 0
            and on_run["state"] == 1
        ):
            complete_episodes.append(
                EpisodeT(
                    start=(
                        index_offset
                        + off_run["start"]
                    ),
                    end=(
                        index_offset
                        + on_run["end"]
                    ),
                    T_OFF=max(
                        minimum_duration,
                        off_run["duration"]
                    ),
                    T_ON=max(
                        minimum_duration,
                        on_run["duration"]
                    ),
                    timezone=timezone,
                )
            )

    if incomplete_policy == "nearest":
        return complete_episodes

    episodes = complete_episodes.copy()

    # The observed/one policies retain their existing boundary treatment.
    def complete_missing_duration(
        observed_duration,
        missing_state,
        local_start,
        local_end
    ):
        if incomplete_policy == "observed":
            return max(minimum_duration, observed_duration)
        return minimum_duration

    # --------------------------------------------------
    # 3. Initial ON block
    # --------------------------------------------------
    first_run = runs[0]

    if first_run["state"] == 1:

        observed_T_ON = max(
            minimum_duration,
            first_run["duration"]
        )

        estimated_T_OFF = (
            complete_missing_duration(
                observed_duration=observed_T_ON,
                missing_state=0,
                local_start=first_run["start"],
                local_end=first_run["end"]
            )
        )

        episodes.append(
            EpisodeT(
                start=(
                    index_offset
                    + first_run["start"]
                ),
                end=(
                    index_offset
                    + first_run["end"]
                ),
                T_OFF=int(estimated_T_OFF),
                T_ON=int(observed_T_ON),
                timezone=timezone,
            )
        )

    # --------------------------------------------------
    # 4. Final OFF block
    # --------------------------------------------------
    last_run = runs[-1]

    if last_run["state"] == 0:

        observed_T_OFF = max(
            minimum_duration,
            last_run["duration"]
        )

        estimated_T_ON = (
            complete_missing_duration(
                observed_duration=observed_T_OFF,
                missing_state=1,
                local_start=last_run["start"],
                local_end=last_run["end"]
            )
        )

        episodes.append(
            EpisodeT(
                start=(
                    index_offset
                    + last_run["start"]
                ),
                end=(
                    index_offset
                    + last_run["end"]
                ),
                T_OFF=int(observed_T_OFF),
                T_ON=int(estimated_T_ON),
                timezone=timezone,
            )
        )

    # Keep chronological order.
    episodes.sort(
        key=lambda episode: episode.start
    )

    return episodes

def validate_binary_profile(X):
    """
    Validate and return a one-dimensional binary profile.
    """

    X = np.asarray(X, dtype=int)

    if X.ndim != 1:
        raise ValueError(
            "X must be one-dimensional."
        )

    if X.size == 0:
        raise ValueError(
            "X cannot be empty."
        )

    if not np.all(np.isin(X, [0, 1])):
        raise ValueError(
            "X must contain only 0 and 1."
        )

    return X


In [ ]:
def compute_ton_toff(
    X,
    index_offset=0,
    incomplete_policy="observed",
    minimum_duration=1,
    operational=False,
    missing_duration=1e-6
):
    """
    Compute T_ON and T_OFF on the daily or operational horizon.

    Operational devices have float durations and NaN outside the first-to-
    last ON window. With no activations, both vectors are entirely NaN.
    A single-state window uses missing_duration for the absent state.

    With nearest, initial ON and final OFF stretches inherit BOTH
    durations from the closest complete OFF -> ON episode. The observed
    ON/OFF profile is preserved. A nonempty profile with no complete
    episode has no nearest reference and raises ValueError for non-operational
    devices. The single-state operational fallback applies to all policies.

    Parameters
    ----------
    X : ndarray
        Binary ON/OFF trajectory.

    Returns
    -------
    T_ON : ndarray
        ON duration assigned to every sample.

    T_OFF : ndarray
        OFF duration assigned to every sample.
    """

    episodes = extract_off_on_episodes(
        X=X,
        index_offset=index_offset,
        incomplete_policy=incomplete_policy,
        minimum_duration=minimum_duration,
        operational=operational,
        missing_duration=missing_duration,
    )

    T = len(X)

    window = get_operation_window(X) if operational else slice(0, T)
    fill_value = np.nan if operational else 0.0
    T_ON = np.full(T, fill_value, dtype=float)
    T_OFF = np.full(T, fill_value, dtype=float)

    for episode in episodes:

        start = episode.start - index_offset
        end = episode.end - index_offset + 1

        T_ON[start:end] = episode.T_ON
        T_OFF[start:end] = episode.T_OFF

    if incomplete_policy == "nearest" and window.stop > window.start:
        if not episodes:
            raise ValueError(
                "Cannot assign nearest T_ON/T_OFF: the profile contains "
                "no complete OFF -> ON episode."
            )

        first = episodes[0]
        initial_end = first.start - index_offset
        T_ON[window.start:initial_end] = first.T_ON
        T_OFF[window.start:initial_end] = first.T_OFF

        last = episodes[-1]
        final_start = last.end - index_offset + 1
        T_ON[final_start:window.stop] = last.T_ON
        T_OFF[final_start:window.stop] = last.T_OFF

    return T_ON, T_OFF

In [ ]:
def compute_time_weight(
    X,
    T_ON,
    T_OFF,
    W_min=1.0,
    W_max=10.0,
    clip=False
):
    """
    Compute the time-flexibility weight.

    During OFF:

        W[t] = W[t-1] + (W_max - W_min) / T_OFF[t]

    During ON:

        W[t] = W[t-1] - (W_max - W_min) / T_ON[t]

    Parameters
    ----------
    X : array-like, shape (T,)
        Binary ON/OFF trajectory.

    T_ON : array-like, shape (T,)
        ON durations.

    T_OFF : array-like, shape (T,)
        OFF durations.

    W_min : float, default=1.0
        Lower reference weight.

    W_max : float, default=10.0
        Upper reference weight.

    clip : bool, default=False
        If True, constrain W to [W_min, W_max].

    Returns
    -------
    W : ndarray of float, shape (T,)
        Time-flexibility weight.
    """

    X = validate_binary_profile(X)

    T_ON = np.asarray(T_ON, dtype=float)
    T_OFF = np.asarray(T_OFF, dtype=float)

    if T_ON.shape != X.shape:
        raise ValueError(
            "T_ON must have the same shape as X."
        )

    if T_OFF.shape != X.shape:
        raise ValueError(
            "T_OFF must have the same shape as X."
        )

    if W_max <= W_min:
        raise ValueError(
            "W_max must be greater than W_min."
        )

    if np.any(T_ON <= 0):
        raise ValueError(
            "T_ON must contain only positive values."
        )

    if np.any(T_OFF <= 0):
        raise ValueError(
            "T_OFF must contain only positive values."
        )

    delta_W = W_max - W_min

    W = np.empty(len(X), dtype=float)

    # Initialization
    if X[0] == 0:
        W[0] = W_min + delta_W / T_OFF[0]
    else:
        W[0] = W_max - delta_W / T_ON[0]

    if clip:
        W[0] = np.clip(
            W[0],
            W_min,
            W_max
        )

    # Evolution
    for t in range(1, len(X)):

        if X[t] == 0:
            W[t] = (
                W[t - 1]
                + delta_W / T_OFF[t]
            )
        else:
            W[t] = (
                W[t - 1]
                - delta_W / T_ON[t]
            )

        if clip:
            W[t] = np.clip(
                W[t],
                W_min,
                W_max
            )

    return W

def compute_power_weight(Y, Y_min, Y_max, W_P_min, W_P_max, m=1e-6):
    Y_safe = np.clip(Y, Y_min, Y_max)

    WP = W_P_min + ((W_P_max - W_P_min) * (Y_safe - Y_min)) / (
        (Y_max - Y_min) + m
    )

    return np.clip(WP, min(W_P_min, W_P_max), max(W_P_min, W_P_max))


In [ ]:
T_ON_FC_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

T_OFF_FC_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

WT_FC_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

WP_FC_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

WNF_FC_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

episodes_by_device = {}
characterization_windows = {}

time_flex_available_matrix = np.zeros(
    (T, N_DEV),
    dtype=bool
)

time_flex_start_matrix = np.zeros(
    (T, N_DEV),
    dtype=bool
)

for j in range(N_DEV):

    if Type[j] == 2:

        X_j = X_onoff_FC_matrix[:, j]

        operational = j in OPERATIONAL_DEVICES

        window = (
            get_operation_window(X_j)
            if operational
            else slice(0, T)
        )

        characterization_windows[j] = window
        time_flex_available_matrix[window, j] = True

        if operational and window.stop > window.start:
            time_flex_start_matrix[window.start, j] = True

        episodes_by_device[j] = extract_off_on_episodes(
            X_j,
            incomplete_policy=INCOMPLETE_POLICY,
            operational=operational,
            missing_duration=MISSING_STATE_DURATION,
        )

        T_ON_j, T_OFF_j = compute_ton_toff(
            X_j,
            incomplete_policy=INCOMPLETE_POLICY,
            operational=operational,
            missing_duration=MISSING_STATE_DURATION,
        )

        T_ON_FC_matrix[:, j] = T_ON_j
        T_OFF_FC_matrix[:, j] = T_OFF_j

        # Para operational devices fica NaN fora da sua
        # janela real de funcionamento.
        WT_FC_matrix[:, j] = np.nan

        if window.stop > window.start:

            WT_FC_matrix[window, j] = compute_time_weight(
                X=X_j[window],
                T_ON=T_ON_j[window],
                T_OFF=T_OFF_j[window],
                clip=False,
            )

    elif Type[j] == 1:

        WP_j = compute_power_weight(
            Y_FC_matrix[:, j],
            Y_min[j],
            Y_max[j],
            Weight_P_min[j],
            Weight_P_max[j],
        )

        WP_FC_matrix[:, j] = np.where(
            X_onoff_FC_matrix[:, j] == 1,
            WP_j,
            np.nan,
        )

    elif Type[j] == 0:

        WNF_FC_matrix[:, j] = 10

## Building Aggregate Weight Evolution Logic

In [ ]:
def compute_WT_from_alpha_beta(
    X_matrix,
    alpha_matrix,
    beta_matrix,
    available_matrix,
    Type,
    block_size
):

    WT_block = np.full(
        (N_BLOCKS, N_DEV),
        np.nan,
        dtype=float
    )

    for j in range(N_DEV):

        if Type[j] != 2:
            continue

        initialized = False
        W_previous = np.nan

        for k in range(N_BLOCKS):

            start = k * block_size
            end = start + block_size

            available = available_matrix[
                start:end,
                j
            ]

            n_available = int(np.sum(available))

            # Fora da janela operational, WT permanece NaN.
            if n_available == 0:
                continue

            X_block = X_matrix[start:end, j]
            X_available = X_block[available]

            if not initialized:

                first_state = int(X_available[0])

                W_previous = (
                    W_MIN
                    if first_state == 0
                    else W_MAX
                )

                initialized = True

            n_on = int(np.sum(X_available))
            n_off = n_available - n_on

            W_current = (
                W_previous
                + n_off * alpha_matrix[k, j]
                - n_on * beta_matrix[k, j]
            )

            WT_block[k, j] = W_current
            W_previous = W_current

    return WT_block

In [ ]:
TF_available_minutes = np.zeros(
    (N_BLOCKS, N_DEV),
    dtype=int
)

TF_block_available = np.zeros(
    (N_BLOCKS, N_DEV),
    dtype=bool
)

alpha_block = np.full(
    (N_BLOCKS, N_DEV),
    np.nan,
    dtype=float
)

beta_block = np.full(
    (N_BLOCKS, N_DEV),
    np.nan,
    dtype=float
)

for j in range(N_DEV):

    if Type[j] != 2:
        continue

    for k in range(N_BLOCKS):

        start = k * BLOCK_SIZE
        end = start + BLOCK_SIZE

        X_block = X_onoff_FC_matrix[start:end, j]
        T_ON_block = T_ON_FC_matrix[start:end, j]
        T_OFF_block = T_OFF_FC_matrix[start:end, j]

        available = time_flex_available_matrix[
            start:end,
            j
        ]

        n_available = int(np.sum(available))

        TF_available_minutes[k, j] = n_available
        TF_block_available[k, j] = n_available > 0

        # Fora da janela operational:
        # alpha e beta permanecem NaN.
        if n_available == 0:
            continue

        valid_ton = (
            available
            & np.isfinite(T_ON_block)
            & (T_ON_block > 0)
        )

        valid_toff = (
            available
            & np.isfinite(T_OFF_block)
            & (T_OFF_block > 0)
        )

        if not np.any(valid_ton):
            raise RuntimeError(
                f"No valid T_ON for device {j}, block {k}."
            )

        if not np.any(valid_toff):
            raise RuntimeError(
                f"No valid T_OFF for device {j}, block {k}."
            )

        # Usar preferencialmente os minutos em que cada taxa
        # intervém na trajetória forecast.
        mask_on = valid_ton & (X_block == 1)
        mask_off = valid_toff & (X_block == 0)

        # Se o bloco só observar um estado, T_ON/T_OFF continua
        # definido pela caracterização do episódio.
        beta_source = (
            mask_on
            if np.any(mask_on)
            else valid_ton
        )

        alpha_source = (
            mask_off
            if np.any(mask_off)
            else valid_toff
        )

        beta_block[k, j] = np.mean(
            9.0 / T_ON_block[beta_source]
        )

        alpha_block[k, j] = np.mean(
            9.0 / T_OFF_block[alpha_source]
        )

In [ ]:
DC_FC_matrix = np.zeros(
    (N_BLOCKS, N_DEV),
    dtype=float
)

for j in range(N_DEV):

    for k in range(N_BLOCKS):

        start = k * BLOCK_SIZE
        end = start + BLOCK_SIZE

        DC_FC_matrix[k, j] = np.mean(
            X_onoff_FC_matrix[start:end, j]
        )


In [ ]:
W_MIN = 1.0
W_MAX = 10.0
DELTA_W = W_MAX - W_MIN

WT_initial = np.full(
    N_DEV,
    np.nan,
    dtype=float
)

for j in range(N_DEV):

    if Type[j] != 2:
        continue

    available_indices = np.flatnonzero(
        time_flex_available_matrix[:, j]
    )

    if len(available_indices) == 0:
        raise RuntimeError(
            f"No available characterization window for device {j}."
        )

    first_available = int(available_indices[0])

    first_state = int(
        X_onoff_FC_matrix[first_available, j]
    )

    WT_initial[j] = (
        W_MIN
        if first_state == 0
        else W_MAX
    )

WT_block_alpha_beta = compute_WT_from_alpha_beta(
    X_matrix=X_onoff_FC_matrix,
    alpha_matrix=alpha_block,
    beta_matrix=beta_block,
    available_matrix=time_flex_available_matrix,
    Type=Type,
    block_size=BLOCK_SIZE,
)

In [ ]:
# ============================================================
# VALIDATION:
# MINUTE-LEVEL WT vs BLOCK-LEVEL WT
# ============================================================
j=2
# Last minute of each scheduling block:
# B=15 -> 14, 29, 44, ..., 1439
block_end_minutes = (
    np.arange(N_BLOCKS) * BLOCK_SIZE
    + BLOCK_SIZE - 1
)

# Minute-level WT evaluated at the end of each block
WT_minute_at_block_end = (
    WT_FC_matrix[block_end_minutes, j]
)

# Block-level WT
WT_block = WT_block_alpha_beta[:, j]

# Difference
error = WT_block - WT_minute_at_block_end

# Time labels
block_end_labels = [
    f"{(m + 1) // 60:02d}:{(m + 1) % 60:02d}"
    if m < TOTAL_MINUTES - 1
    else "24:00"
    for m in block_end_minutes
]

# --------------------------------------------------------
# TABLE
# --------------------------------------------------------

df_validation = pd.DataFrame({
    "Block": np.arange(1, N_BLOCKS + 1),
    "End_time": block_end_labels,
    "WT_minute": WT_minute_at_block_end,
    "WT_block": WT_block,
    "Error": error,
    "Abs_error": np.abs(error),
    "DC_FC": DC_FC_matrix[:, j],
})

print("\n" + "=" * 100)
print(
    f"WT VALIDATION — Device {j}: "
    f"{dev_names[j]}"
)
print("=" * 100)

print(
    df_validation.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nError summary:")
print(
    f"Mean absolute error: "
    f"{np.mean(np.abs(error)):.6f}"
)
print(
    f"Maximum absolute error: "
    f"{np.max(np.abs(error)):.6f}"
)
print(
    f"Final error: "
    f"{error[-1]:.6f}"
)

# --------------------------------------------------------
# PLOT
# --------------------------------------------------------

plt.figure(figsize=(15, 6))

# Original minute-level WT
plt.plot(
    np.arange(T),
    WT_FC_matrix[:, j],
    label="WT minute-level",
    linewidth=1.5
)

# Block-level WT located at the end of each block
plt.plot(
    block_end_minutes,
    WT_block,
    marker="o",
    markersize=3,
    linewidth=1.5,
    label=f"WT block-level (B={BLOCK_SIZE} min)"
)

plt.xlabel("Minute of day")
plt.ylabel(r"$W_j^T$")
plt.title(
    f"Minute vs block WT — "
    f"{dev_names[j]}"
)

plt.xticks(
    np.arange(0, TOTAL_MINUTES + 1, 120),
    [
        f"{h:02d}:00"
        for h in range(0, 25, 2)
    ],
    rotation=45
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BUILD BLOCK-LEVEL INPUT DATAFRAME FOR OPTIMIZATION
# ============================================================

rows = []

for k in range(N_BLOCKS):

    start = k * BLOCK_SIZE
    end = start + BLOCK_SIZE

    # identify timezone of this block
    block_start_minute = start

    timezone_k = next(
        tz
        for tz, limits in TIME_ZONES.items()
        if limits["start"] <= block_start_minute < limits["end"]
    )

    for j in range(N_DEV):

        X_block = X_onoff_FC_matrix[start:end, j]
        Y_block = Y_FC_matrix[start:end, j]

        row = {
            "block": k,
            "block_start_min": start,
            "block_end_min": end,
            "timezone": timezone_k,

            "appliance_j": j,
            "device": DEVICE_NAMES[j],
            "Type": int(Type[j]),

            "Power_max_W": float(Power_max[j]),

            # baseline operational quantities
            "DC_FC": float(np.mean(X_block)),
            "n_on_FC": float(np.sum(X_block)),

            # useful for power-flexible devices
            "Y_FC": float(np.mean(Y_block)),
        }

        # ----------------------------------------------------
        # TIME-FLEXIBLE PARAMETERS
        # ----------------------------------------------------
        if Type[j] == 2:

            row["alpha"] = (
                float(alpha_block[k, j])
                if TF_block_available[k, j]
                else 0.0
            )

            row["beta"] = (
                float(beta_block[k, j])
                if TF_block_available[k, j]
                else 0.0
            )
            
            row["WT_initial"] = float(WT_initial[j])

            row["WT_FC_end"] = (
                float(WT_block_alpha_beta[k, j])
                if TF_block_available[k, j]
                else np.nan
            )

        else:

            row["alpha"] = np.nan
            row["beta"] = np.nan
            row["WT_initial"] = np.nan
            row["WT_FC_end"] = np.nan

        # ----------------------------------------------------
        # POWER-FLEXIBLE PARAMETERS
        # ----------------------------------------------------
        if Type[j] == 1:

            row["Y_min"] = float(Y_min[j])
            row["Y_max"] = float(Y_max[j])

        else:

            row["Y_min"] = np.nan
            row["Y_max"] = np.nan

        # ----------------------------------------------------
        # ENERGY LIMIT
        # same value repeated for all devices in the block
        # ----------------------------------------------------
        row["E_limit_Wh"] = get_e_lim_k(k)
        row["TF_available_minutes"] = int(
            TF_available_minutes[k, j]
        )

        row["TF_block_available"] = bool(
            TF_block_available[k, j]
        )

        rows.append(row)


optimization_input_df = pd.DataFrame(rows)
optimization_input_df["price_EUR_kWh"] = (
    optimization_input_df["timezone"]
    .map(TOU_PRICE_EUR_KWH)
)

# Day-ahead planning

## Models

### Objetivo de utilidade: desvios absolutos à previsão

A função objetivo minimiza:

$$
\lambda_T \sum_{j\in J_T,k\in K} W^{T,FC}_{j,k}\left|n^{FC}_{j,k}-n_{j,k}\right|
+ \lambda_P \sum_{j\in J_P,k\in K} W^{P,FC}_{j,k}\left|Y^{FC,ON}_{j,k}-Y_{j,k}\right|
+ \frac{\lambda_{rlx}}{|K|}\sum_{k\in K}\frac{E^{rlx}_k}{E^{lim}_k}.
$$

O módulo é aplicado a cada diferença, antes da soma: cortes e operação acima da previsão têm penalização positiva e não se anulam entre blocos. Isto mede desvio à previsão, não apenas corte de serviço.

Para manter o modelo linear e compatível com o HiGHS, cada módulo é representado por uma variável não negativa `d`, com `d >= previsão - decisão` e `d >= decisão - previsão`. Quando o coeficiente é positivo, minimizar o objetivo faz `d` coincidir com o módulo.

`WT_ref`, `WP_ref` e `Y_ref` são referências fixas da previsão nos minutos ON. Os pesos dos blocos sem atividade prevista continuam a ser zero: nesses blocos, os desvios não contribuem para o objetivo. O termo temporal usa minutos e o de potência usa frações de potência; `lambda_T_DA` e `lambda_P_DA` controlam a importância relativa.

`E_rlx` mantém o tratamento original: variável não negativa, não fixada, somada ao limite de energia de cada bloco. A penalização mantém a divisão por `E_lim[k]` e pelo número de blocos, com `lambda_rlx = 100.0`. O módulo elimina contribuições negativas, mas não garante relaxação zero.

Após o solve, a tabela `cost_breakdown` apresenta as mesmas expressões usadas pelo objetivo.


Solver diagnostics are held in memory. Each solve uses the time limit and MIP-gap target stated in its call. Physical metrics use unrounded decisions in kWh and EUR; a feasible time-limited solution does not imply proven optimality.

In [ ]:
# ============================================================
# 1. STORAGE VARIABLES / MODEL INPUTS
# ============================================================

df = optimization_input_df.copy()

# -------------------------
# Sets
# -------------------------
blocks = sorted(df["block"].unique())

J_T = sorted(
    df.loc[df["Type"] == 2, "appliance_j"].unique()
)

J_P = sorted(
    df.loc[df["Type"] == 1, "appliance_j"].unique()
)

J_NF = sorted(
    df.loc[df["Type"] == 0, "appliance_j"].unique()
)

ordered_blocks = sorted(blocks)

first_block = ordered_blocks[0]

previous_block = {
    ordered_blocks[i]: ordered_blocks[i - 1]
    for i in range(1, len(ordered_blocks))
}

# -------------------------
# Time-flexibility parameters
# -------------------------

alpha = {
    (j, k): float(
        df.loc[
            (df["appliance_j"] == j)
            & (df["block"] == k),
            "alpha"
        ].iloc[0]
    )
    for j in J_T
    for k in blocks
}

beta = {
    (j, k): float(
        df.loc[
            (df["appliance_j"] == j)
            & (df["block"] == k),
            "beta"
        ].iloc[0]
    )
    for j in J_T
    for k in blocks
}

W0 = {
    j: float(
        df.loc[
            df["appliance_j"] == j,
            "WT_initial"
        ].dropna().iloc[0]
    )
    for j in J_T
}

WP_min_model = {
    j: float(Weight_P_min[j])
    for j in J_P
}

WP_max_model = {
    j: float(Weight_P_max[j])
    for j in J_P
}

# -------------------------
# Device parameters
# -------------------------

Pmax = {
    j: float(Power_max[j])
    for j in range(N_DEV)
}

Ymin_model = {
    j: float(Y_min[j])
    for j in J_P
}

Ymax_model = {
    j: float(Y_max[j])
    for j in J_P
}
# -------------------------
# Baseline block parameters
# -------------------------

DC_FC = {
    (j, k): float(
        df.loc[
            (df["appliance_j"] == j)
            & (df["block"] == k),
            "DC_FC"
        ].iloc[0]
    )
    for j in range(N_DEV)
    for k in blocks
}

n_on_FC = {
    (j, k): DC_FC[j, k] * BLOCK_SIZE
    for j in J_T
    for k in blocks
}


# -------------------------
# Non-flexible energy
# Fixed energy per block
# -------------------------

E_NF = {
    k: sum(
        Pmax[j]
        * DC_FC[j, k]
        * BLOCK_SIZE
        / 60.0
        for j in J_NF
    )
    for k in blocks
}

TF_available = {
    (j, k): int(
        df.loc[
            (df["appliance_j"] == j)
            & (df["block"] == k),
            "TF_available_minutes",
        ].iloc[0]
    )
    for j in J_T
    for k in blocks
}


# -------------------------
# Energy limits
# -------------------------

E_lim = {
    k: float(
        df.loc[
            df["block"] == k,
            "E_limit_Wh"
        ].iloc[0]
    )
    for k in blocks
}
price = {
    k: float(
        df.loc[df.block == k, "price_EUR_kWh"].iloc[0]
    )
    for k in blocks
}

# ------------------------------------------------------------
# Fixed forecast references for utility (active minutes only)
# ------------------------------------------------------------
# Averaging over active minutes avoids diluting the reference
# power fraction with OFF samples. Inactive blocks have zero
# reference weights and therefore no contribution to the objective.
WT_ref = {}
WP_ref = {}
Y_ref = {}

for k in blocks:
    start = k * BLOCK_SIZE
    end = start + BLOCK_SIZE

    for j in J_T:
        active = X_onoff_FC_matrix[start:end, j] == 1
        WT_ref[j, k] = (
            float(np.mean(WT_FC_matrix[start:end, j][active]))
            if np.any(active) else 0.0
        )

    for j in J_P:
        active = X_onoff_FC_matrix[start:end, j] == 1
        WP_ref[j, k] = (
            float(np.mean(WP_FC_matrix[start:end, j][active]))
            if np.any(active) else 0.0
        )
        Y_ref[j, k] = (
            float(np.mean(Y_FC_matrix[start:end, j][active]))
            if np.any(active) else Ymin_model[j]
        )


In [ ]:
from pathlib import Path

# ============================================================
# MODELO 1 — pesos forecast do bloco anterior
# ============================================================

lambda_T_DA = 1.0
lambda_P_DA = 1.0
lambda_rlx = 1000.0
EPSILON_P = 1e-4


# Mesma relação Y -> WP usada nas restrições.
def power_weight(j, y):
    return WP_min_model[j] + (
        (WP_max_model[j] - WP_min_model[j])
        / (Ymax_model[j] - Ymin_model[j] + EPSILON_P)
    ) * (y - Ymin_model[j])


# Peso inicial de potência, comum aos dois modelos.
WP_initial_model = {
    j: power_weight(j, Y_ref[j, first_block])
    for j in J_P
}

# Pesos fixos usados pelo objetivo do modelo 1.
# WT_block_alpha_beta contém o estado forecast no FIM do bloco.
WT_input_1 = {}

for j in J_T:

    for k in blocks:

        # Fora da janela de disponibilidade não há custo temporal.
        if TF_available[j, k] == 0:
            WT_input_1[j, k] = 0.0
            continue

        if k == first_block:
            WT_input_1[j, k] = W0[j]
            continue

        k_previous = previous_block[k]
        previous_WT = WT_block_alpha_beta[
            k_previous,
            j
        ]

        # Primeiro bloco operacional: o bloco anterior ainda
        # pode estar fora da janela e, portanto, ser NaN.
        WT_input_1[j, k] = (
            float(previous_WT)
            if np.isfinite(previous_WT)
            else W0[j]
        )

WP_input_1 = {
    (j, k): (
        WP_initial_model[j]
        if k == first_block
        else power_weight(j, Y_ref[j, previous_block[k]])
    )
    for j in J_P
    for k in blocks
}

# ------------------------------------------------------------
# Model and sets
# ------------------------------------------------------------

m = pe.ConcreteModel()

m.K = pe.Set(initialize=blocks, ordered=True)
m.JT = pe.Set(initialize=J_T)
m.JP = pe.Set(initialize=J_P)
m.JNF = pe.Set(initialize=J_NF)

# Parâmetros: fixos durante cada solve.
# mutable=True permite atualizá-los no segundo modelo.
m.W_time = pe.Param(
    m.JT,
    m.K,
    initialize=WT_input_1,
    mutable=True,
    within=pe.NonNegativeReals
)

m.W_power = pe.Param(
    m.JP,
    m.K,
    initialize=WP_input_1,
    mutable=True,
    within=pe.NonNegativeReals
)

# ------------------------------------------------------------
# Decision variables
# ------------------------------------------------------------

def n_on_bounds(m, j, k):
    return 0, TF_available[j, k]


m.n_on = pe.Var(
    m.JT,
    m.K,
    domain=pe.NonNegativeIntegers,
    bounds=n_on_bounds,
)

m.Y_rec = pe.Var(
    m.JP,
    m.K,
    bounds=lambda m, j, k: (
        Ymin_model[j],
        Ymax_model[j]
    )
)

# Estados resultantes das decisões desta resolução.
m.WT = pe.Var(m.JT, m.K, bounds=(1.0, 10.0))
m.WP = pe.Var(m.JP, m.K)

# Relaxação original: não negativa e não fixada.
m.E_rlx = pe.Var(m.K, domain=pe.NonNegativeReals)

# ------------------------------------------------------------
# Time-flexibility dynamics
# ------------------------------------------------------------

def WT_evolution_rule(m, j, k):

    W_previous = (
        W0[j]
        if k == first_block
        else m.WT[j, previous_block[k]]
    )

    return (
       m.WT[j, k] 
       ==
        W_previous
        + TF_available[j, k] * alpha[j, k]
        - m.n_on[j, k] * (
            alpha[j, k] + beta[j, k]
        )
    )


m.WT_evolution = pe.Constraint(
    m.JT,
    m.K,
    rule=WT_evolution_rule
)

# ------------------------------------------------------------
# Power-flexibility weight
# ------------------------------------------------------------

m.WP_definition = pe.Constraint(
    m.JP,
    m.K,
    rule=lambda m, j, k: (
        m.WP[j, k] == power_weight(j, m.Y_rec[j, k])
    )
)

# ------------------------------------------------------------
# Energy limit
# ------------------------------------------------------------

def energy_limit_rule(m, k):

    E_TF = sum(
        Pmax[j] * m.n_on[j, k] / 60.0
        for j in m.JT
    )

    E_PF = sum(
        Pmax[j]
        * m.Y_rec[j, k]
        * DC_FC[j, k]
        * BLOCK_SIZE
        / 60.0
        for j in m.JP
    )

    return (
        E_NF[k] + E_TF + E_PF
        <= E_lim[k] + m.E_rlx[k]
    )


m.energy_limit = pe.Constraint(
    m.K,
    rule=energy_limit_rule
)

# ------------------------------------------------------------
# Utility deviations
# ------------------------------------------------------------

# Minutes removed relative to the forecast:
# n_on_FC - n_on > 0
m.n_on_deviation_below = pe.Var(
    m.JT,
    m.K,
    domain=pe.NonNegativeReals
)

# Minutes added relative to the forecast:
# n_on - n_on_FC > 0
m.n_on_deviation_above = pe.Var(
    m.JT,
    m.K,
    domain=pe.NonNegativeReals
)


# Exact decomposition:
#
# n_on - n_on_FC
# =
# added minutes - removed minutes
def time_deviation_balance_rule(m, j, k):

    return (
        m.n_on[j, k]
        - n_on_FC[j, k]
        ==
        m.n_on_deviation_above[j, k]
        - m.n_on_deviation_below[j, k]
    )


m.time_deviation_balance = pe.Constraint(
    m.JT,
    m.K,
    rule=time_deviation_balance_rule
)


# Power reduction relative to the forecast.
# Its non-negative domain also imposes Y_rec <= Y_ref.
m.Y_deviation = pe.Var(
    m.JP,
    m.K,
    domain=pe.NonNegativeReals
)


def power_deviation_rule(m, j, k):

    return (
        m.Y_deviation[j, k]
        ==
        Y_ref[j, k]
        - m.Y_rec[j, k]
    )


m.power_deviation = pe.Constraint(
    m.JP,
    m.K,
    rule=power_deviation_rule
)


# ------------------------------------------------------------
# Objective
#
# W_time/W_power:
# fixed weights used by each optimization.
#
# WT/WP:
# flexibility states resulting from the current decisions.
# ------------------------------------------------------------

m.time_deviation_cost = pe.Expression(
    expr=sum(

        # Removing forecast ON minutes:
        # cost increases with W_T.
        m.W_time[j, k]
        * m.n_on_deviation_below[j, k]

        +

        # Adding ON minutes:
        # cost decreases with W_T.
        (
            W_MAX
            - m.W_time[j, k]
        )
        * m.n_on_deviation_above[j, k]

        for j in m.JT
        for k in m.K
    )
)


m.power_deviation_cost = pe.Expression(
    expr=sum(
        m.W_power[j, k]
        * m.Y_deviation[j, k]

        for j in m.JP
        for k in m.K
    )
)


m.relaxation_cost = pe.Expression(
    expr=sum(
        m.E_rlx[k] / 1000.0
        for k in m.K
    )
)


def objective_rule(m):

    return (
        lambda_T_DA
        * m.time_deviation_cost

        +

        lambda_P_DA
        * m.power_deviation_cost

        +

        lambda_rlx
        * m.relaxation_cost
    )


m.objective = pe.Objective(
    rule=objective_rule,
    sense=pe.minimize
)

# ------------------------------------------------------------
# Helpers reutilizados pelo modelo 2
# ------------------------------------------------------------



def diagnostic_operational_metrics(model):
    return day_ahead_operational_metrics(
        model, Pmax=Pmax, DC_FC=DC_FC, E_NF=E_NF,
        price=price, block_size=BLOCK_SIZE,
    )


def report_costs(model):

    table = pd.DataFrame({
        "Componente": ["Tempo", "Potência", "Relaxação"],
        "Valor sem lambda": [
            pe.value(model.time_deviation_cost),
            pe.value(model.power_deviation_cost),
            pe.value(model.relaxation_cost),
        ],
        "Lambda": [
            lambda_T_DA,
            lambda_P_DA,
            lambda_rlx,
        ],
    })

    table["Contribuição no objetivo"] = (
        table["Valor sem lambda"] * table["Lambda"]
    )

    print(table.to_string(index=False, float_format="%.6f"))
    print(f"Objetivo: {pe.value(model.objective):.6f}")

    print(
        "E_rlx total: "
        f"{sum(pe.value(model.E_rlx[k]) for k in model.K) / 1000:.6f} kWh"
    )

    return table

# ============================================================
# SOLVE MODEL 1
# ============================================================

solver_1, result_1, diagnostics_1 = solve_day_ahead(
    m,
    scenario_name="Max Utility 1",
    operational_metrics=diagnostic_operational_metrics,
    time_limit=60,
    mip_rel_gap=0.01,
    tee=True,
)

# Preservar o modelo e os resultados da primeira resolução.
m1 = m
day_ahead_feasible = diagnostics_1["feasible_solution"]
solver, result = solver_1, result_1
cost_breakdown_1 = cost_breakdown = None
W_Opt_T, W_Opt_P, n_on_rec_1, Y_rec_1 = {}, {}, {}, {}
n_on_rec_rounded, DC_rec_rounded = {}, {}
if day_ahead_feasible:
    cost_breakdown_1 = report_costs(m1)

    # Pesos otimizados no FIM de cada bloco.
    # São números, não referências às variáveis Pyomo.
    W_Opt_T = {
        (j, k): pe.value(m1.WT[j, k])
        for j in J_T
        for k in blocks
    }

    W_Opt_P = {
        (j, k): pe.value(m1.WP[j, k])
        for j in J_P
        for k in blocks
    }

    n_on_rec_1 = {
        (j, k): pe.value(m1.n_on[j, k])
        for j in J_T
        for k in blocks
    }

    Y_rec_1 = {
        (j, k): pe.value(m1.Y_rec[j, k])
        for j in J_P
        for k in blocks
    }

    # Compatibilidade com as células seguintes do notebook.
    solver, result = solver_1, result_1
    cost_breakdown = cost_breakdown_1

    n_on_rec_rounded = {
        key: int(round(value))
        for key, value in n_on_rec_1.items()
    }

    DC_rec_rounded = {
        key: value / BLOCK_SIZE
        for key, value in n_on_rec_rounded.items()
    }
else:
    print("Model 1 has no feasible incumbent; Model 2 and dependent reporting will be skipped.")


In [ ]:
# Model 2 must never run without a feasible Model 1 incumbent.
m2 = solver_2 = result_2 = cost_breakdown_2 = None
W_Opt_T_2, W_Opt_P_2, n_on_rec_2, Y_rec_2 = {}, {}, {}, {}
if diagnostics_1["feasible_solution"]:
    # ============================================================
    # MODELO 2 — pesos otimizados no modelo 1
    # ============================================================

    # Cópia independente:
    # mesmas variáveis, restrições, referências e estados iniciais.
    m2 = m1.clone()


    # ------------------------------------------------------------
    # Atualizar os pesos usados pelo objetivo
    # ------------------------------------------------------------

    for j in J_T:

        for k in blocks:

            # Fora de Delta_j não existe custo de flexibilidade
            # temporal.
            if TF_available[j, k] == 0:

                m2.W_time[j, k] = 0.0

            elif k == first_block:

                m2.W_time[j, k] = W0[j]

            else:

                m2.W_time[j, k] = W_Opt_T[
                    j,
                    previous_block[k]
                ]


    for j in J_P:

        for k in blocks:

            m2.W_power[j, k] = (
                WP_initial_model[j]
                if k == first_block
                else W_Opt_P[
                    j,
                    previous_block[k]
                ]
            )


    # ------------------------------------------------------------
    # Solve: voltar a otimizar o dia inteiro
    # ------------------------------------------------------------

    solver_2, result_2, diagnostics_2 = solve_day_ahead(
        m2,
        scenario_name="Max Utility 2",
        operational_metrics=diagnostic_operational_metrics,
        time_limit=60,
        mip_rel_gap=0.01,
        tee=True,
    )

    day_ahead_feasible = diagnostics_2["feasible_solution"]
    m = m2
    solver, result = solver_2, result_2
    cost_breakdown = None
    n_on_rec_rounded, DC_rec_rounded = {}, {}
    if day_ahead_feasible:
        cost_breakdown_2 = report_costs(m2)


        # ------------------------------------------------------------
        # Guardar os novos estados e decisões
        # ------------------------------------------------------------

        W_Opt_T_2 = {
            (j, k): pe.value(m2.WT[j, k])
            for j in J_T
            for k in blocks
        }

        W_Opt_P_2 = {
            (j, k): pe.value(m2.WP[j, k])
            for j in J_P
            for k in blocks
        }

        n_on_rec_2 = {
            (j, k): pe.value(m2.n_on[j, k])
            for j in J_T
            for k in blocks
        }

        Y_rec_2 = {
            (j, k): pe.value(m2.Y_rec[j, k])
            for j in J_P
            for k in blocks
        }


        # ------------------------------------------------------------
        # As células seguintes passam a usar o modelo 2
        # ------------------------------------------------------------

        m = m2
        solver, result = solver_2, result_2
        cost_breakdown = cost_breakdown_2


        n_on_rec_rounded = {
            key: int(round(value))
            for key, value in n_on_rec_2.items()
        }

        DC_rec_rounded = {
            key: value / BLOCK_SIZE
            for key, value in n_on_rec_rounded.items()
        }


        # ------------------------------------------------------------
        # Alteração das decisões entre as duas resoluções
        # ------------------------------------------------------------

        print(
            "Maior alteração de n_on entre modelos:",
            max(
                (
                    abs(
                        n_on_rec_2[key]
                        - n_on_rec_1[key]
                    )
                    for key in n_on_rec_1
                ),
                default=0.0,
            ),
            "min",
        )


        print(
            "Maior alteração de Y_rec entre modelos:",
            max(
                (
                    abs(
                        Y_rec_2[key]
                        - Y_rec_1[key]
                    )
                    for key in Y_rec_1
                ),
                default=0.0,
            ),
        )
    else:
        print("Model 2 has no feasible incumbent; dependent reporting is skipped.")
else:
    day_ahead_feasible = False
    diagnostics_2 = skipped_diagnostics(
        "Max Utility 2", "Model 1 did not produce a feasible incumbent."
    )

# The objectives use different weights; their values are not directly comparable.


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    # ============================================================
    # BUILD SCHEDULING RESULTS DATAFRAME
    # ============================================================

    schedule_results = []

    for k in blocks:

        start_minute = k * BLOCK_SIZE

        time_label = (
            f"{start_minute // 60:02d}:"
            f"{start_minute % 60:02d}"
        )

        for j in range(N_DEV):

            row = {
                "block": k,
                "time": time_label,
                "appliance_j": j,
                "appliance_name": DEVICE_NAMES[j],
                "Type": int(Type[j]),
                "DC_FC": DC_FC[j, k],
            }

            # ====================================================
            # TIME-FLEXIBLE
            # ====================================================

            if j in J_T:

                n_on_FC_block = (
                    DC_FC[j, k]
                    * BLOCK_SIZE
                )

                n_on_rec = pe.value(
                    m.n_on[j, k]
                )

                row.update({
                    "n_on_FC": n_on_FC_block,
                    "n_on_rec": n_on_rec_rounded[j,k],

                    "DC_rec": DC_rec_rounded[j,k],

                    "WT_FC": (
                        WT_block_alpha_beta[k, j]
                    ),

                    "WT_rec": pe.value(
                        m.WT[j, k]
                    ),
                })

            else:

                row.update({
                    "n_on_FC": np.nan,
                    "n_on_rec": np.nan,
                    "DC_rec": np.nan,
                    "WT_FC": np.nan,
                    "WT_rec": np.nan,
                })

            # ====================================================
            # POWER-FLEXIBLE
            # ====================================================

            if j in J_P:

                Y_FC_block = float(
                    df.loc[
                        (df["appliance_j"] == j)
                        & (df["block"] == k),
                        "Y_FC"
                    ].iloc[0]
                )

                # Modulação durante os minutos ON, do modelo final.
                Y_rec_on = pe.value(m2.Y_rec[j, k])

                # Média do bloco, incluindo os minutos OFF.
                Y_rec = Y_rec_on * DC_FC[j, k]

                row.update({
                    "Y_FC": Y_FC_block,
                    "Y_rec": Y_rec,
                    "Y_FC_on": Y_ref[j, k],
                    "Y_rec_on": Y_rec_on,
                    "WP_rec": pe.value(m2.WP[j, k]),
                })
            else:

                row.update({
                    "Y_FC": np.nan,
                    "Y_rec": np.nan,
                    "WP_rec": np.nan,
                    "Y_FC_on": np.nan,
                    "Y_rec_on": np.nan,
                })

            schedule_results.append(row)


    schedule_results_df = pd.DataFrame(
        schedule_results
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    # ============================================================
    # DAY-AHEAD -> REAL-TIME INTERFACE
    #
    # Block-level flexibility-weight targets:
    #
    # WT_DA_block_matrix[k, j]
    #     -> Day-Ahead WT target for time-flexible device j
    #
    # WP_DA_block_matrix[k, j]
    #     -> Day-Ahead WP target for power-flexible device j
    # ============================================================

    WT_DA_block_matrix = np.full(
        (N_BLOCKS, N_DEV),
        np.nan,
        dtype=float
    )

    WP_DA_block_matrix = np.full(
        (N_BLOCKS, N_DEV),
        np.nan,
        dtype=float
    )


    # ============================================================
    # TIME-FLEXIBLE TARGETS
    # ============================================================

    df_WT_DA = (
        schedule_results_df[
            schedule_results_df["Type"] == 2
        ]
        .copy()
    )

    for _, row in df_WT_DA.iterrows():

        k = int(row["block"])
        j = int(row["appliance_j"])

        WT_DA_block_matrix[
            k,
            j
        ] = float(
            row["WT_rec"]
        )
    
    # Operational devices have no Day-Ahead WT target
    # outside their operational block window.
    for j in OPERATIONAL_DEVICES:

        unavailable_blocks = (
            ~TF_block_available[:, j]
        )

        WT_DA_block_matrix[
            unavailable_blocks,
            j
        ] = np.nan


    # ============================================================
    # POWER-FLEXIBLE TARGETS
    # ============================================================

    df_WP_DA = (
        schedule_results_df[
            schedule_results_df["Type"] == 1
        ]
        .copy()
    )

    for _, row in df_WP_DA.iterrows():

        k = int(row["block"])
        j = int(row["appliance_j"])

        WP_DA_block_matrix[
            k,
            j
        ] = float(
            row["WP_rec"]
        )


    # ============================================================
    # VALIDATION
    # ============================================================

    print(
        "WT_DA_block_matrix shape:",
        WT_DA_block_matrix.shape
    )

    print(
        "WP_DA_block_matrix shape:",
        WP_DA_block_matrix.shape
    )

    print(
        "Expected:",
        (N_BLOCKS, N_DEV)
    )


    # Time-flexible devices
    for j in J_T:

        required_blocks = (
            TF_block_available[:, j]
        )

        missing_required_targets = (
            required_blocks
            & np.isnan(
                WT_DA_block_matrix[:, j]
            )
        )

        if np.any(missing_required_targets):

            missing_blocks = np.flatnonzero(
                missing_required_targets
            )

            raise ValueError(
                f"Missing active Day-Ahead WT target "
                f"for device {j} ({DEVICE_NAMES[j]}), "
                f"blocks {missing_blocks.tolist()}."
            )


    # Power-flexible devices
    for j in J_P:

        if np.isnan(
            WP_DA_block_matrix[:, j]
        ).any():

            raise ValueError(
                f"Missing Day-Ahead WP target "
                f"for power-flexible device {j} "
                f"({DEVICE_NAMES[j]})."
            )
else:
    print("Skipped: no feasible day-ahead solution.")


# Day-ahead plots and metrics

In [ ]:
# Historical B=10/15/30 results omitted here: current-run metrics are shown below.

In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    energy_results = []

    for k in blocks:

        # --------------------------------------------------------
        # Recommended energy
        # --------------------------------------------------------
        E_TF_rec = sum(
            Pmax[j] * n_on_rec_rounded[j, k] / 60.0
            for j in J_T
        )

        E_PF_rec = sum(
            Pmax[j]
            * pe.value(m.Y_rec[j, k])
            * DC_FC[j, k]
            * BLOCK_SIZE
            / 60.0
            for j in J_P
        )

        E_total_rec = E_NF[k] + E_TF_rec + E_PF_rec
        E_total_model = E_NF[k] + E_PF_rec + sum(
            Pmax[j] * pe.value(m.n_on[j, k]) / 60.0
            for j in J_T
        )
        rounded_violation_Wh = max(E_total_rec - E_lim[k], 0.0)

        # --------------------------------------------------------
        # Baseline energy
        # --------------------------------------------------------
        E_TF_base = sum(
            Pmax[j]
            * DC_FC[j, k]
            * BLOCK_SIZE
            / 60.0
            for j in J_T
        )

        E_PF_base = sum(
            Pmax[j]
            * float(
                df.loc[
                    (df["appliance_j"] == j)
                    & (df["block"] == k),
                    "Y_FC",
                ].iloc[0]
            )
            * BLOCK_SIZE
            / 60.0
            for j in J_P
        )

        E_baseline = E_NF[k] + E_TF_base + E_PF_base
        delta_E = E_total_rec - E_baseline

        baseline_violation_Wh = max(E_baseline - E_lim[k], 0.0)

        # --------------------------------------------------------
        # Store results
        # --------------------------------------------------------
        energy_results.append({
            "block": k,
            "timezone": df.loc[df["block"] == k, "timezone"].iloc[0] + 1,
            "E_baseline_Wh": E_baseline,
            "E_total_rec_Wh": E_total_rec,
            "E_total_model_Wh": E_total_model,
            "rounded_violation_Wh": rounded_violation_Wh,
            "Delta_E_Wh": delta_E,
            "E_limit_Wh": E_lim[k],
            "baseline_violation_Wh": baseline_violation_Wh,
            "E_NF_Wh": E_NF[k],
            "E_TF_base_Wh": E_TF_base,
            "E_PF_base_Wh": E_PF_base,
            "E_TF_rec_Wh": E_TF_rec,
            "E_PF_rec_Wh": E_PF_rec,
            "E_rlx_Wh": pe.value(m.E_rlx[k]),
        })

    energy_results_df = pd.DataFrame(energy_results)
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    # ============================================================
    # TOTAL ENERGY - 24 HOURS
    # ============================================================

    E_baseline_total = energy_results_df["E_baseline_Wh"].sum()
    E_rec_total = energy_results_df["E_total_rec_Wh"].sum()
    E_rlx_total = energy_results_df["E_rlx_Wh"].sum()

    print("\n" + "=" * 50)
    print("TOTAL ENERGY - 24 HOURS")
    print("=" * 50)

    print(f"Baseline energy:     {E_baseline_total / 1000:.3f} kWh")
    print(f"Recommended energy:  {E_rec_total / 1000:.3f} kWh")
    print(f"Total relaxation:    {E_rlx_total / 1000:.3f} kWh")

    print("=" * 50)
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
# Physical metrics use block energy in Wh and the same tariff in EUR/kWh.
# A feasible incumbent is reported even when the 60 s time limit is reached.
if day_ahead_feasible:
    from IPython.display import display

    solve_records = [diagnostics_1, diagnostics_2]
    final_diagnostics = solve_records[-1]
    planning_time_s = sum(d["execution_time_s"] for d in solve_records)
    tolerance_reached = all(
        d["feasible_solution"] and np.isfinite(d["mip_gap_pct"])
        and d["mip_gap_pct"] <= 100.0 * d["mip_rel_gap_target"] + 1e-9
        for d in solve_records
    )
    block_prices = energy_results_df["block"].map(price)
    assert block_prices.notna().all(), "Missing block tariff"

    def block_metrics(energy_column):
        energy_Wh = energy_results_df[energy_column]
        excess_Wh = (energy_Wh - energy_results_df["E_limit_Wh"]).clip(lower=0.0)
        return {
            "Energia [kWh]": energy_Wh.sum() / 1000.0,
            "Custo [EUR]": (block_prices * energy_Wh).sum() / 1000.0,
            "Blocos violados": int((excess_Wh > 1e-6).sum()),
            "Excesso total [kWh]": excess_Wh.sum() / 1000.0,
            "Excesso máximo [Wh]": excess_Wh.max(),
        }

    baseline_metrics = block_metrics("E_baseline_Wh")
    baseline_row = {
        "Plano": "Baseline", **baseline_metrics,
        "Variação energia [%]": 0.0, "Variação custo [%]": 0.0,
        "Variação excesso [%]": 0.0,
        "Terminação": "N/A", "Gap final [%]": np.nan,
        "Tolerância 5% atingida": "N/A", "Tempo DA [s]": np.nan,
    }
    comparison = []
    for label, energy_column in [
        ("Day-ahead contínuo", "E_total_model_Wh"),
        ("Day-ahead arredondado", "E_total_rec_Wh"),
    ]:
        metrics = block_metrics(energy_column)
        row = {"Plano": label, **metrics}
        for variation, metric in [
            ("Variação energia [%]", "Energia [kWh]"),
            ("Variação custo [%]", "Custo [EUR]"),
            ("Variação excesso [%]", "Excesso total [kWh]"),
        ]:
            baseline_value = baseline_metrics[metric]
            row[variation] = (100.0 * (metrics[metric] / baseline_value - 1.0)
                              if baseline_value != 0 else np.nan)
        row.update({
            "Terminação": " / ".join(d["termination_condition"] for d in solve_records),
            "Gap final [%]": final_diagnostics["mip_gap_pct"],
            "Tolerância 5% atingida": "Sim" if tolerance_reached else "Não",
            "Tempo DA [s]": planning_time_s,
        })
        comparison.append(row)

    comparison_df = pd.DataFrame([baseline_row, *comparison])
    solver_performance_df = pd.DataFrame([
        {
            "Resolução": d["scenario_name"],
            "Limite [s]": d["time_limit_s"],
            "Tempo [s]": d["execution_time_s"],
            "Terminação": d["termination_condition"],
            "Gap final [%]": d["mip_gap_pct"],
            "Tolerância 5% atingida": (
                "Sim" if d["feasible_solution"] and np.isfinite(d["mip_gap_pct"])
                and d["mip_gap_pct"] <= 5.0 + 1e-9 else "Não"
            ),
        } for d in solve_records
    ])

    # Verify the notebook's block sums against the solver's in-memory metrics.
    np.testing.assert_allclose(
        [comparison[0]["Energia [kWh]"], comparison[0]["Custo [EUR]"]],
        [final_diagnostics["energy_consumption_kWh"], final_diagnostics["energy_cost_EUR"]],
        rtol=1e-10, atol=1e-8,
    )
    model_energy_Wh = np.array([
        pe.value(m.energy_limit[k].body) + E_lim[k] + pe.value(m.E_rlx[k])
        for k in energy_results_df["block"]
    ])
    np.testing.assert_allclose(model_energy_Wh, energy_results_df["E_total_model_Wh"],
                               rtol=1e-10, atol=1e-6)
    model_excess_Wh = np.maximum(model_energy_Wh - energy_results_df["E_limit_Wh"], 0.0)
    assert int((model_excess_Wh > 1e-6).sum()) == comparison[0]["Blocos violados"]
    np.testing.assert_allclose(model_excess_Wh.sum() / 1000.0,
                               comparison[0]["Excesso total [kWh]"], rtol=1e-10, atol=1e-8)

    print(f"B={BLOCK_SIZE} min | lambda_rlx={lambda_rlx:g} | 60 s por otimização | tolerância=5%")
    print("Energia, custo e violações: valores por bloco verificados contra o modelo.")
    print("Variações: 100*(DA/baseline - 1); valores negativos representam reduções.")
    display(comparison_df.round(6))
    print(comparison_df.to_string(index=False, float_format="%.6f", na_rep="N/A"))
    display(solver_performance_df.round(6))
    print(f"Tempo total de planeamento (soma das resoluções): {planning_time_s:.6f} s")
    print("Tempos incluem transferência APPSI e resolução. O gap refere-se ao objetivo MILP, não ao custo de energia.")

    # Keep all plot inputs, including both Max Utility stages and raw decisions.
    energy_export_df = energy_results_df.copy()
    energy_export_df["rho_EUR_kWh"] = block_prices
    for prefix, column in [("baseline", "E_baseline_Wh"), ("model", "E_total_model_Wh"),
                            ("rounded", "E_total_rec_Wh")]:
        energy_export_df[prefix + "_cost_EUR"] = block_prices * energy_export_df[column] / 1000.0
        excess = (energy_export_df[column] - energy_export_df["E_limit_Wh"]).clip(lower=0.0)
        energy_export_df[prefix + "_exceedance_kWh"] = excess / 1000.0
        energy_export_df[prefix + "_violated"] = excess > 1e-6
    model_stages = [("model_1", m1), ("model_2", m2)]
    device_results_by_stage = {}
    for stage_name, stage_model in model_stages:
        devices_export_df = optimization_input_df.copy()
        for name, component, eligible in [
            ("n_on_model_min", stage_model.n_on, J_T),
            ("Y_model_on", stage_model.Y_rec, J_P),
            ("WT_model_end", stage_model.WT, J_T),
            ("WP_model", stage_model.WP, J_P),
            ("WT_objective_weight", stage_model.W_time, J_T),
            ("WP_objective_weight", stage_model.W_power, J_P),
        ]:
            devices_export_df[name] = [
                pe.value(component[j, k]) if j in eligible else np.nan
                for j, k in zip(devices_export_df.appliance_j, devices_export_df.block)
            ]
        for name, reference in [("WT_forecast_active_mean", WT_ref),
                                 ("WP_forecast_active_mean", WP_ref), ("Y_forecast_on", Y_ref)]:
            devices_export_df[name] = [reference.get((j, k), np.nan)
                for j, k in zip(devices_export_df.appliance_j, devices_export_df.block)]
        device_results_by_stage[stage_name] = devices_export_df.copy()
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# Baseline is available independently of optimization feasibility.
baseline_device_energy_Wh = (
    X_onoff_FC_matrix * Y_FC_matrix * Power_max[np.newaxis, :]
).reshape(N_BLOCKS, BLOCK_SIZE, N_DEV).sum(axis=1) / 60.0
fig_baseline, ax_baseline = plot_block_energy_stacked(
    baseline_device_energy_Wh, dev_names, E_lim_reference_vector,
    block_size=BLOCK_SIZE, tick_step=120, colors=colors,
    title="Baseline consumption",
)

# A valid day-ahead solution is required for the recommended schedule.
if day_ahead_feasible:
    schedule_blocks = energy_results_df.sort_values("block")["block"].to_numpy(dtype=int)
    recommended_device_energy_Wh = baseline_device_energy_Wh[schedule_blocks].copy()
    for row, k in enumerate(schedule_blocks):
        for j in J_T:
            recommended_device_energy_Wh[row, j] = Pmax[j] * n_on_rec_rounded[j, k] / 60.0
        for j in J_P:
            recommended_device_energy_Wh[row, j] = (
                Pmax[j] * pe.value(m.Y_rec[j, k]) * DC_FC[j, k] * BLOCK_SIZE / 60.0
            )
    fig_schedule, ax_schedule = plot_schedule(
        energy_results_df,
        block_size=BLOCK_SIZE,
        tick_step=120,
        title="Day-Ahead Scheduling Results",
        colors=colors,
        device_energy_Wh=recommended_device_energy_Wh,
        device_names=dev_names,
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    heatmap_result = plot_time_flexible_activation_heatmap(
        schedule_results_df,
        block_size=BLOCK_SIZE,
        tick_step=120,
        reference_ax=ax_schedule
    )
    if heatmap_result is not None:
        fig_heatmap, ax_heatmap = heatmap_result
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    fig_power, ax_power = plot_power_flexible_behavior(
        schedule_results_df,
        device="Light1",
        block_size=BLOCK_SIZE,
        tick_step=120
    )
else:
    print("Skipped: no feasible day-ahead solution.")


## Energy-reduction contribution by flexibility type

In [ ]:
# Decompose the aggregate energy reduction by flexibility type.
from pathlib import Path


def compute_energy_reduction_by_flexibility(devices_df, blocks_df, block_size):
    """Return baseline-minus-plan energy and validate it against block totals."""
    expected_blocks = 1440 // block_size
    expected_devices = devices_df["appliance_j"].nunique()
    if len(blocks_df) != expected_blocks:
        raise ValueError(
            f"Expected {expected_blocks} blocks for B={block_size}, "
            f"received {len(blocks_df)}."
        )
    if len(devices_df) != expected_blocks * expected_devices:
        raise ValueError("Device results do not match the selected block duration.")

    tf = devices_df.loc[devices_df["Type"] == 2].copy()
    pf = devices_df.loc[devices_df["Type"] == 1].copy()
    tf_reduction = float((
        (tf["Power_max_W"] * tf["n_on_FC"] / 60.0).sum()
        - (tf["Power_max_W"] * tf["n_on_model_min"] / 60.0).sum()
    ) / 1000.0)
    pf_reduction = float((
        (pf["Power_max_W"] * pf["Y_FC"] * block_size / 60.0).sum()
        - (pf["Power_max_W"] * pf["Y_model_on"] * pf["DC_FC"]
           * block_size / 60.0).sum()
    ) / 1000.0)
    total_reduction = float(
        (blocks_df["E_baseline_Wh"].sum()
         - blocks_df["E_total_model_Wh"].sum()) / 1000.0
    )
    np.testing.assert_allclose(
        tf_reduction + pf_reduction, total_reduction, rtol=0, atol=1e-9
    )
    return {
        "Time-flexible [kWh]": tf_reduction,
        "Power-flexible [kWh]": pf_reduction,
        "Total reduction [kWh]": total_reduction,
    }


def plot_energy_reduction_by_flexibility(strategy_results, block_size):
    """Plot the validated energy-reduction contribution of each flexibility type."""
    rows = []
    for strategy, (devices_df, blocks_df) in strategy_results.items():
        rows.append({
            "Strategy": strategy,
            **compute_energy_reduction_by_flexibility(
                devices_df, blocks_df, block_size
            ),
        })
    reduction_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(13.5, 6.0), constrained_layout=True)
    x = np.arange(len(reduction_df))
    positive_bottom = np.zeros(len(reduction_df))
    negative_bottom = np.zeros(len(reduction_df))
    components = [
        ("Time-flexible [kWh]", "Time-flexible devices", "#4F81BD"),
        ("Power-flexible [kWh]", "Power-flexible devices", "#70AD47"),
    ]
    for column, label, color in components:
        values = reduction_df[column].to_numpy()
        bottoms = np.where(values >= 0, positive_bottom, negative_bottom)
        bars = ax.bar(
            x, values, bottom=bottoms, width=0.58,
            color=color, label=label, zorder=3,
        )
        for bar, value, bottom in zip(bars, values, bottoms):
            if abs(value) >= 0.05:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bottom + value / 2,
                    f"{value:.2f}",
                    ha="center", va="center", fontsize=12,
                    color="white" if abs(value) > 0.35 else "black",
                )
        positive_bottom += np.where(values >= 0, values, 0.0)
        negative_bottom += np.where(values < 0, values, 0.0)

    for position, total, stack_top in zip(
        x, reduction_df["Total reduction [kWh]"], positive_bottom
    ):
        ax.annotate(
            f"Total: {total:.2f} kWh",
            xy=(position, stack_top), xytext=(0, 8),
            textcoords="offset points", ha="center", va="bottom",
            fontsize=12, fontweight="bold",
        )

    ax.axhline(0, color="0.25", lw=0.8)
    ax.set_xticks(x, reduction_df["Strategy"])
    ax.set_ylabel("Energy reduction relative to baseline [kWh]", fontsize=16)
    ax.set_title(
        "Contribution to total energy reduction by flexibility type",
        fontsize=18,
    )
    ax.tick_params(axis="both", labelsize=14)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    ax.margins(y=0.18)
    ax.legend(loc="upper left", ncol=2, frameon=True, fontsize=14)
    return fig, ax, reduction_df


if day_ahead_feasible:
    _consumption = run_day_ahead_for_comparison("Day_Ahead_Plan_Min_Consumption.ipynb", BLOCK_SIZE)
    _cost = run_day_ahead_for_comparison("Day_Ahead_Plan_Min_Cost.ipynb", BLOCK_SIZE)
    _strategy_results = {
        "DA Max Utility": (device_results_by_stage["model_2"], energy_export_df),
        "DA Min Consumption": (_consumption["device_results_by_stage"]["final"], _consumption["energy_export_df"]),
        "DA Min Cost": (_cost["device_results_by_stage"]["final"], _cost["energy_export_df"]),
    }
    energy_reduction_figure, energy_reduction_axis, energy_reduction_df = plot_energy_reduction_by_flexibility(
        _strategy_results, block_size=BLOCK_SIZE,
    )
    display(energy_reduction_df.round(6))
    plt.show()
else:
    print("Skipped: no feasible day-ahead solution.")


# Integrated real-time control

## Real-time model

In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    # =====================================================
    # RT INPUT FROM DAY-AHEAD PLANNING
    # =====================================================

    X_plan_matrix = X_onoff_FC_matrix.copy()
    Y_plan_matrix = Y_FC_matrix.copy()

    X_plan_ref_matrix = X_onoff_FC_matrix.copy()
    Y_plan_ref_matrix = Y_FC_matrix.copy()
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    # ============================================================
    # MAX UTILITY — STORAGE
    # ============================================================

    X_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=int
    )

    Y_rt_matrix = np.ones(
        (T, N_DEV),
        dtype=float
    )

    W_T_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=float
    )

    W_P_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=float
    )

    W_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=float
    )

    P_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=float
    )

    P_rt_sum = np.zeros(
        T,
        dtype=float
    )


    force_on_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=int
    )

    count_hold_matrix = np.zeros(
        (T, N_DEV),
        dtype=int
    )

    P_lim_vector = np.zeros(
        T,
        dtype=float
    )

    P_rlx_vector = np.zeros(
        T,
        dtype=float
    )

    Optimization_flag = np.zeros(
        T,
        dtype=int
    )

    P_before_sum = np.zeros(
        T,
        dtype=float
    )
    force_off_rt_matrix = np.zeros(
        (T, N_DEV),
        dtype=int
    )

    prevent_off_vector = np.zeros(
        N_DEV,
        dtype=int,
    )

    # Day-Ahead WT target used at each RT minute
    W_T_DA_target_rt_matrix = np.full(
        (T, N_DEV),
        np.nan,
        dtype=float
    )

    # Predicted WT inside the optimization
    # after applying X_opt
    W_T_predicted_rt_matrix = np.full(
        (T, N_DEV),
        np.nan,
        dtype=float
    )

    # Final tracking error after execution
    W_T_DA_error_rt_matrix = np.full(
        (T, N_DEV),
        np.nan,
        dtype=float
    )
    W_P_DA_target_rt_matrix = np.full(
        (T, N_DEV),
        np.nan,
        dtype=float
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    def get_w_max(j, minute, variable_hysteresis=False):

        if variable_hysteresis:

            # Drinks Refrigerator
            if j == 0:

                # 13:00 -> 15:00
                if 22*60+30 <= minute < 24*60:
                    return 20.0
    
        return 10.0

    def function_dataset(
        i,
        X_RT_matrix,
        W_T_RT_matrix,
        X_plan_matrix,
        Type,
        time_flex_available_matrix,
        time_flex_start_matrix,
        variable_hysteresis=False
    ):

        X_RT_vector = X_plan_matrix[
            i,
            :
        ].copy()

        force_on_vector = np.zeros(
            len(Type),
            dtype=int
        )

        force_off_vector = np.zeros(
            len(Type),
            dtype=int
        )

        if i == 0:
            return (
                X_RT_vector,
                force_on_vector,
                force_off_vector
            )


        for j in range(len(Type)):

            if Type[j] == 2:

                # Preserve the reference outside the window. On entry there is
                # no previous in-window state to use for threshold overrides.
                if (not time_flex_available_matrix[i, j]
                        or time_flex_start_matrix[i, j]):
                    continue

                previous_W = float(
                    W_T_RT_matrix[
                        i - 1,
                        j
                    ]
                )

                if not np.isfinite(previous_W):
                    raise ValueError(f"Missing active RT weight at minute {i}, device {j}.")

                W_max_now = get_w_max(
                    j,
                    i,
                    variable_hysteresis
                )


                # ============================================
                # MAXIMUM ON TIME REACHED
                # ============================================

                if previous_W <= 1.0:
                    X_RT_vector[j] = 0
                    force_off_vector[j] = 1
                # ============================================
                # MAXIMUM OFF TIME REACHED
                # ============================================
                elif previous_W >= W_max_now:
                    X_RT_vector[j] = 1
                    force_on_vector[j] = 1


                # ============================================
                # NO OPERATIONAL OVERRIDE
                # ============================================

                else:

                    X_RT_vector[j] = (
                        X_plan_matrix[
                            i,
                            j
                        ]
                    )


            else:

                X_RT_vector[j] = (
                    X_plan_matrix[
                        i,
                        j
                    ]
                )


        return (
            X_RT_vector,
            force_on_vector,
            force_off_vector
        )
    
    def update_power_before_optimization(
        X_before_opt_rt_vector,
        Y_plan_vector,
        Y_prev_opt_rt_vector,
        count_prev_vector,
        Type,
    ):
        """
        Build Y before RT optimization.

        Type 1:
            Follow Day-Ahead Y_plan when no power-hold constraint is active.

            If still inside T_Hold_Power:
                keep previous executed Y.

        Other types:
            Y = 1.
        """

        Y_before_opt_rt_vector = np.ones(
            len(Type),
            dtype=float
        )

        count_vector = count_prev_vector.copy()

        for j in range(len(Type)):

            # Device is OFF
            if X_before_opt_rt_vector[j] == 0:

                Y_before_opt_rt_vector[j] = 1.0
                count_vector[j] = 0


            # ============================================
            # POWER-FLEXIBLE
            # ============================================

            elif Type[j] == 1:

                if count_vector[j] > 0:

                    # Still within power hold
                    Y_before_opt_rt_vector[j] = (
                        Y_prev_opt_rt_vector[j]
                    )

                    count_vector[j] -= 1

                else:

                    # Follow Day-Ahead power recommendation
                    Y_before_opt_rt_vector[j] = (
                        Y_plan_vector[j]
                    )


            # ============================================
            # TYPE 0 / TYPE 2
            # ============================================

            else:

                Y_before_opt_rt_vector[j] = 1.0
                count_vector[j] = 0


        return (
            Y_before_opt_rt_vector,
            count_vector
        )

    def update_power_hold_after_optimization(
        X_opt_rt_vector,
        Y_before_opt_rt_vector,
        Y_opt_rt_vector,
        count_vector,
        Type,
        TimeHoldPower,
    ):
        count_updated = count_vector.copy()

        for j in range(len(Type)):

            if Type[j] == 1 and X_opt_rt_vector[j] == 1:

                if abs(Y_opt_rt_vector[j] - Y_before_opt_rt_vector[j]) > 1e-6:
                    count_updated[j] = int(TimeHoldPower[j])

            else:
                count_updated[j] = 0

        return count_updated

    def time_weight_delta(state, t_on, t_off):
        """One-minute weight change; preserve positive sub-minute durations."""
        if not (np.isfinite(t_on) and np.isfinite(t_off)
                and t_on > 0 and t_off > 0):
            raise ValueError("Active time-flexible durations must be finite and positive.")
        return -9.0 / t_on if state == 1 else 9.0 / t_off


    def initialize_time_weights_from_state(
        X0,
        Type,
        T_ON_0,
        T_OFF_0,
        time_flex_available,
    ):
        W_T0 = np.full(len(X0), np.nan, dtype=float)

        for j in range(len(X0)):
            if Type[j] != 2 or not time_flex_available[j]:
                continue

            W_T0[j] = 10.0 if X0[j] == 1 else 1.0

        return W_T0


    def prepare_time_weights_for_minute(
        prev_W_T,
        X_reference,
        Type,
        time_flex_available,
        time_flex_start,
    ):
        """Prepare pre-decision weights, seeding each operational window once.

        Availability comes from the reference window, not from NaN comparisons.
        A missing weight inside an ongoing window is an error.
        """
        weights = np.full(len(Type), np.nan, dtype=float)
        for j in range(len(Type)):
            if Type[j] != 2 or not time_flex_available[j]:
                continue
            if time_flex_start[j]:
                weights[j] = 10.0 if X_reference[j] == 1 else 1.0
            elif np.isfinite(prev_W_T[j]):
                weights[j] = prev_W_T[j]
            else:
                raise ValueError(f"Missing weight inside active window for device {j}.")
        return weights

    def compute_power_vector(X_vec, Power_max, Y_vec):
        """
        Compute device power vector.

        P[j] = Power_max[j] * Y[j] * X[j]
        """
        return Power_max * Y_vec * X_vec

    def compute_power_sum(P_vec):
        """
        Total power at the current minute.
        """
        return float(np.sum(P_vec))

    def update_time_weights(
        prev_W_T,
        X_now,
        Type,
        T_ON_now,
        T_OFF_now,
        minute,
        time_flex_available,
        variable_hysteresis=False,
    ):
        """Update available time weights without clipping or duration rounding.

        Outside the operating window weights remain NaN. Within it, reactive
        thresholds are still applied by function_dataset on the next minute.
        minute and variable_hysteresis retain their existing interface role.
        """
        W_T_new = np.full(len(X_now), np.nan, dtype=float)
        for j in range(len(X_now)):
            if Type[j] != 2 or not time_flex_available[j]:
                continue
            if not np.isfinite(prev_W_T[j]):
                raise ValueError(f"Missing active weight at minute {minute}, device {j}.")
            W_T_new[j] = prev_W_T[j] + time_weight_delta(
                X_now[j], T_ON_now[j], T_OFF_now[j]
            )
        return W_T_new
    def compute_wp_vector(Y_vec, Weight_P_min, Weight_P_max, Y_min, Y_max):
        epsilon = 1e-4

        return (
            Weight_P_min
            +
            (
                (Weight_P_max - Weight_P_min)
                /
                (Y_max - Y_min + epsilon)
            )
            * (Y_vec - Y_min)
        )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    m = pe.ConcreteModel()

    # =========================
    # Sets
    # =========================
    m.J = pe.RangeSet(0, N_DEV - 1)

    # =========================
    # Fixed parameters
    # =========================
    m.Type = pe.Param(m.J, initialize={j: int(Type[j]) for j in range(N_DEV)})
    m.P_max = pe.Param(m.J, initialize={j: float(Power_max[j]) for j in range(N_DEV)})

    m.Y_min = pe.Param(m.J, initialize={j: float(Y_min[j]) for j in range(N_DEV)})
    m.Y_max = pe.Param(m.J, initialize={j: float(Y_max[j]) for j in range(N_DEV)})

    m.W_P_min = pe.Param(m.J, initialize={j: float(Weight_P_min[j]) for j in range(N_DEV)})
    m.W_P_max = pe.Param(m.J, initialize={j: float(Weight_P_max[j]) for j in range(N_DEV)})

    m.T_Hold_Power = pe.Param(
        m.J,
        initialize={j: float(TimeHoldPower[j]) for j in range(N_DEV)}
    )


    # =========================
    # Mutable RT parameters
    # updated each minute
    # =========================
    m.P_lim = pe.Param(initialize=0.0, mutable=True)

    m.X_in = pe.Param(m.J, initialize=0, mutable=True)
    m.X_prev = pe.Param(m.J, initialize=0, mutable=True)

    m.W_time = pe.Param(m.J, initialize=0.0, mutable=True)
    m.W_power = pe.Param(m.J,initialize=1.0,mutable=True)



    # Y actual antes da otimização
    m.Y_in = pe.Param(m.J, initialize={j: float(Y_min[j]) for j in range(N_DEV)}, mutable=True)

    # contador do tempo que falta / já passou desde alteração de potência
    m.count_T_Hold_Power = pe.Param(m.J, initialize=0, mutable=True)

    m.force_on = pe.Param(m.J, initialize=0, mutable=True)
    m.force_off = pe.Param(m.J, initialize=0, mutable=True)

    m.time_flex_available = pe.Param(
        m.J,
        initialize=0,
        mutable=True,
        within=pe.Binary,
    )
    m.prevent_off = pe.Param(
        m.J,
        initialize=0,
        mutable=True,
        within=pe.Binary,
    )

    # ============================================================
    # DAY-AHEAD WT TRACKING PARAMETERS
    # ============================================================

    # Day-Ahead WT target associated with the current block k
    m.W_DA = pe.Param(
        m.J,
        initialize=0.0,
        mutable=True
    )
    m.WP_DA = pe.Param(
        m.J,
        initialize=0.0,
        mutable=True
    )

    # Candidate WT if the RT controller executes X_opt = 1
    m.W_if_on = pe.Param(
        m.J,
        initialize=0.0,
        mutable=True
    )


    # Candidate WT if the RT controller executes X_opt = 0
    m.W_if_off = pe.Param(
        m.J,
        initialize=0.0,
        mutable=True
    )

    m.W_power_pred = pe.Var(
        m.J,
        domain=pe.NonNegativeReals
    )

    m.W_power_deviation = pe.Var(
        m.J,
        domain=pe.NonNegativeReals
    )

    # =========================
    # Decision variables
    # =========================
    m.X_opt = pe.Var(m.J, domain=pe.Binary)
    m.Y_opt = pe.Var(m.J, domain=pe.NonNegativeReals)
    m.Y_active = pe.Var(m.J, domain=pe.NonNegativeReals)
    m.P_rlx = pe.Var(domain=pe.NonNegativeReals)

    # Predicted time-flexibility weight resulting from X_opt
    m.W_time_pred = pe.Var(m.J,domain=pe.Reals)
    # Absolute deviation between predicted RT WT and Day-Ahead WT target
    m.W_time_deviation = pe.Var(m.J,domain=pe.NonNegativeReals)

    # =========================
    # Constraints
    # =========================

    def power_limit_rule(m):
        return sum(
            m.P_max[j] * m.Y_active[j]
            for j in m.J
        ) <= m.P_lim + m.P_rlx

    m.power_limit = pe.Constraint(rule=power_limit_rule)

    def y_active_upper_x_rule(m, j):
        return m.Y_active[j] <= m.Y_max[j] * m.X_opt[j]

    def y_active_upper_y_rule(m, j):
        return m.Y_active[j] <= m.Y_opt[j]

    def y_active_lower_rule(m, j):
        return m.Y_active[j] >= (
            m.Y_opt[j] - m.Y_max[j] * (1 - m.X_opt[j])
        )

    m.y_active_upper_x = pe.Constraint(m.J, rule=y_active_upper_x_rule)
    m.y_active_upper_y = pe.Constraint(m.J, rule=y_active_upper_y_rule)
    m.y_active_lower = pe.Constraint(m.J, rule=y_active_lower_rule)

    def prevent_weight_overshoot_rule(m, j):

        if pe.value(m.Type[j]) != 2:
            return pe.Constraint.Skip

        return (
            m.X_opt[j]
            >=
            m.prevent_off[j]
        )


    m.prevent_weight_overshoot = pe.Constraint(
        m.J,
        rule=prevent_weight_overshoot_rule,
    )


    # Y lower bound
    def y_min_rule(m, j):
        return m.Y_opt[j] >= m.Y_min[j]

    m.y_min_constraint = pe.Constraint(m.J, rule=y_min_rule)


    # Y upper bound
    def y_max_rule(m, j):
        return m.Y_opt[j] <= m.Y_max[j]

    m.y_max_constraint = pe.Constraint(m.J, rule=y_max_rule)

    # ---------------------------
    # Constraint 2:
    # Status logic by flexibility type
    #
    # Type 0 = non-flexible  -> X_opt = X_in
    # Type 1 = power-flexible -> X_opt = X_in
    # Type 2 = time-flexible  -> X_opt = 0 or 1
    #
    # ---------------------------
    def status_logic_rule(m, j):

        device_type = int(
            pe.value(m.Type[j])
        )

        # ========================================================
        # TIME-FLEXIBLE DEVICE
        # ========================================================

        if device_type == 2:

            # Fora da janela de flexibilidade:
            # seguir obrigatoriamente a referência.
            if int(
                pe.value(m.time_flex_available[j])
            ) == 0:

                return (
                    m.X_opt[j]
                    ==
                    m.X_in[j]
                )

            # Tempo máximo desligado atingido:
            # equipamento obrigatoriamente ligado.
            if int(
                pe.value(m.force_on[j])
            ) == 1:

                return (
                    m.X_opt[j]
                    ==
                    1
                )

            # Tempo máximo ligado atingido:
            # equipamento obrigatoriamente desligado.
            if int(
                pe.value(m.force_off[j])
            ) == 1:

                return (
                    m.X_opt[j]
                    ==
                    0
                )

            # Dentro da janela de flexibilidade:
            # X_opt é livre para assumir 0 ou 1.
            #
            # Como X_opt já tem domínio Binary, não é necessária
            # nenhuma restrição adicional.
            return pe.Constraint.Feasible

        # ========================================================
        # NON-FLEXIBLE AND POWER-FLEXIBLE DEVICES
        # ========================================================

        if device_type in [0, 1]:

            return (
                m.X_opt[j]
                ==
                m.X_in[j]
            )

        return pe.Constraint.Feasible

    m.status_logic = pe.Constraint(
        m.J,
        rule=status_logic_rule
    )

    def y_rt_upper_rule(m, j):

        if pe.value(m.Type[j]) == 1:
            return m.Y_opt[j] <= m.Y_in[j]

        return pe.Constraint.Skip


    m.y_rt_upper = pe.Constraint(
        m.J,
        rule=y_rt_upper_rule
    )

    # ============================================================
    # PREDICTED WT AFTER RT DECISION
    #
    # X_opt = 0:
    #     W_pred = W_if_off
    #
    # X_opt = 1:
    #     W_pred = W_if_on
    #
    # Because X_opt is binary, this expression exactly selects
    # the corresponding candidate WT.
    # ============================================================

    def wt_prediction_rule(m, j):

        if pe.value(m.Type[j]) != 2:

            return (
                m.W_time_pred[j]
                == 0.0
            )

        return (
            m.W_time_pred[j]
            ==
            m.W_if_off[j]
            +
            (
                m.W_if_on[j]
                -
                m.W_if_off[j]
            )
            *
            m.X_opt[j]
        )


    m.wt_prediction_constraint = pe.Constraint(
        m.J,
        rule=wt_prediction_rule
    )

    # ============================================================
    # ABSOLUTE DAY-AHEAD WT TRACKING ERROR
    #
    # W_time_deviation >= W_pred - W_DA
    # W_time_deviation >= W_DA - W_pred
    #
    # Since W_time_deviation is minimized, it becomes:
    #
    # |W_pred - W_DA|
    # ============================================================

    def wt_deviation_positive_rule(m, j):

        if pe.value(m.Type[j]) != 2:

            return (
                m.W_time_deviation[j]
                == 0.0
            )

        return (
            m.W_time_deviation[j]
            >=
            m.W_time_pred[j]
            -
            m.W_DA[j]
        )


    def wt_deviation_negative_rule(m, j):

        if pe.value(m.Type[j]) != 2:

            return pe.Constraint.Skip

        return (
            m.W_time_deviation[j]
            >=
            m.W_DA[j]
            -
            m.W_time_pred[j]
        )


    m.wt_deviation_positive = pe.Constraint(
        m.J,
        rule=wt_deviation_positive_rule
    )

    m.wt_deviation_negative = pe.Constraint(
        m.J,
        rule=wt_deviation_negative_rule
    )

    def wp_prediction_rule(m, j):

        if pe.value(m.Type[j]) != 1:
            return m.W_power_pred[j] == 0.0

        slope = (
            (m.W_P_max[j] - m.W_P_min[j])
            /
            (
                m.Y_max[j]
                - m.Y_min[j]
                + EPSILON_P
            )
        )

        return (
            m.W_power_pred[j]
            ==
            m.W_P_min[j]
            +
            slope
            *
            (
                m.Y_opt[j]
                -
                m.Y_min[j]
            )
        )


    m.wp_prediction_constraint = pe.Constraint(
        m.J,
        rule=wp_prediction_rule
    )

    def wp_deviation_positive_rule(m, j):

        if pe.value(m.Type[j]) != 1:
            return m.W_power_deviation[j] == 0.0

        return (
            m.W_power_deviation[j]
            >=
            m.W_power_pred[j]
            -
            m.WP_DA[j]
        )


    def wp_deviation_negative_rule(m, j):

        if pe.value(m.Type[j]) != 1:
            return pe.Constraint.Skip

        return (
            m.W_power_deviation[j]
            >=
            m.WP_DA[j]
            -
            m.W_power_pred[j]
        )


    m.wp_deviation_positive = pe.Constraint(
        m.J,
        rule=wp_deviation_positive_rule
    )

    m.wp_deviation_negative = pe.Constraint(
        m.J,
        rule=wp_deviation_negative_rule
    )

    def power_hold_rule(m, j):
        # se não for power-flexible, mantém Y = Y_in
        if pe.value(m.Type[j]) != 1:
            return m.Y_opt[j] == m.Y_in[j]

        # se ainda está dentro do T_Hold_Power, mantém o Y anterior
        if pe.value(m.count_T_Hold_Power[j]) != 0:
            return m.Y_opt[j] == m.Y_in[j]

        # caso contrário, pode otimizar Y dentro de [Y_min, Y_max]
        return pe.Constraint.Feasible

    m.power_hold_constraint = pe.Constraint(m.J, rule=power_hold_rule)

    # ============================================================
    # OBJECTIVE FUNCTION
    # ============================================================

    lambda_DA_WT = 1.0
    lambda_DA_WP = 1.0
    lambda_rlx = 100000.0

    def objective_rule(m):

        time_DA_tracking = sum(
            m.W_time_deviation[j]
            for j in m.J
            if pe.value(m.Type[j]) == 2
        )

        power_DA_tracking = sum(
            m.X_in[j] *
            m.W_power_deviation[j]
            for j in m.J
            if pe.value(m.Type[j]) == 1
        )

        return (
            lambda_DA_WT
            * time_DA_tracking

            +

            lambda_DA_WP
            * power_DA_tracking
        
            + lambda_rlx * m.P_rlx

        )


    m.objective = pe.Objective(
        rule=objective_rule,
        sense=pe.minimize
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:

    # ============================================================
    # MAX UTILITY RT LOOP
    # ============================================================

    from time import perf_counter

    rt_iteration_time_s = np.full(T, np.nan, dtype=float)
    rt_solver_time_s = np.full(T, np.nan, dtype=float)

    solver_utility = pe.SolverFactory("highs")

    variable_hysteresis = False

    # ============================================================
    # INITIAL STATE
    # ============================================================

    X_initial = X_onoff_FC_matrix[0, :].copy()

    W_T_initial = initialize_time_weights_from_state(
        X0=X_initial,
        Type=Type,
        T_ON_0=T_ON_FC_matrix[0, :],
        T_OFF_0=T_OFF_FC_matrix[0, :],
        time_flex_available=time_flex_available_matrix[0, :],
    )

    count_initial = np.zeros(
        N_DEV,
        dtype=int
    )

    # ============================================================
    # MINUTE-BY-MINUTE RT CONTROL
    # ============================================================

    for i in range(T):

        rt_iteration_started = perf_counter()

        # ========================================================
        # STEP 1 — CURRENT INPUTS
        # ========================================================

        P_lim = get_p_lim_for_minute(i)

        T_ON_now = T_ON_FC_matrix[i, :].copy()
        T_OFF_now = T_OFF_FC_matrix[i, :].copy()
        time_flex_available_now = (time_flex_available_matrix[i, :])

        X_plan_vector = X_plan_ref_matrix[i, :].copy()
        Y_plan_vector = Y_plan_ref_matrix[i, :].copy()

        P_lim_vector[i] = P_lim
    
        current_block = i // BLOCK_SIZE

        WT_DA_target_vector = (
            WT_DA_block_matrix[
                current_block,
                :
            ].copy()
        )
        # Exact minute-level mask. This is necessary for partially
        # available boundary blocks.
        for j in J_T:

            if not time_flex_available_now[j]:
                WT_DA_target_vector[j] = np.nan
        W_T_DA_target_rt_matrix[
            i,
            :
        ] = WT_DA_target_vector
        WP_DA_target_vector = (
            WP_DA_block_matrix[
                current_block,
                :
            ].copy()
        )
        W_P_DA_target_rt_matrix[
            i,
            :
        ] = WP_DA_target_vector

        # ========================================================
        # STEP 2 — PREVIOUS RT STATE
        # ========================================================

        if i == 0:

            X_previous = X_initial.copy()

            Y_previous = np.ones(
                N_DEV,
                dtype=float
            )

            W_T_previous = W_T_initial.copy()

            count_previous = count_initial.copy()

        else:

            X_previous = X_rt_matrix[
                i - 1, :
            ].copy()

            Y_previous = Y_rt_matrix[
                i - 1, :
            ].copy()

            W_T_previous = W_T_rt_matrix[
                i - 1, :
            ].copy()

            count_previous = count_hold_matrix[
                i - 1, :
            ].copy()
    
        W_T_previous = prepare_time_weights_for_minute(
            prev_W_T=W_T_previous,
            X_reference=X_plan_vector,
            Type=Type,
            time_flex_available=time_flex_available_now,
            time_flex_start=time_flex_start_matrix[i, :],
        )

        # ========================================================
        # STEP 3 — TIME-FLEXIBLE PRE-PROCESSING
        #
        # Build X_before and operational overrides.
        # ========================================================

        if i == 0:

            X_before = X_plan_vector.copy()

            force_on_vector = np.zeros(
                N_DEV,
                dtype=int
            )

            force_off_vector = np.zeros(
                N_DEV,
                dtype=int
            )

        else:

            (
            X_before,
            force_on_vector,
            force_off_vector
            ) = function_dataset(
                i=i,
                X_RT_matrix=X_rt_matrix,
                W_T_RT_matrix=W_T_rt_matrix,
                X_plan_matrix=X_plan_ref_matrix,
                Type=Type,
                time_flex_available_matrix=time_flex_available_matrix,
                time_flex_start_matrix=time_flex_start_matrix,
                variable_hysteresis=variable_hysteresis,
            )

        # ========================================================
        # STEP 4 — POWER-FLEXIBLE PRE-PROCESSING
        # ========================================================

        (
            Y_before,
            count_vector
        ) = update_power_before_optimization(

            X_before_opt_rt_vector=X_before,

            Y_plan_vector=Y_plan_vector,

            Y_prev_opt_rt_vector=Y_previous,

            count_prev_vector=count_previous,

            Type=Type
        )

        # ========================================================
        # STEP 5 — POWER-FLEXIBILITY WEIGHT
        # ========================================================

        W_P_before = compute_wp_vector(

            Y_vec=Y_before,

            Weight_P_min=Weight_P_min,

            Weight_P_max=Weight_P_max,

            Y_min=Y_min,

            Y_max=Y_max
        )

        # ========================================================
        # OPTIONAL — POWER BEFORE RT OPTIMIZATION
        # ========================================================

        P_before = compute_power_vector(
            X_vec=X_before,
            Power_max=Power_max,
            Y_vec=Y_before
        )

        P_before_sum[i] = compute_power_sum(
            P_before
        )
    
        prevent_off_vector = np.zeros(
            N_DEV,
            dtype=int,
        )

        for j in range(N_DEV):

            if (
                Type[j] == 2
                and time_flex_available_now[j]
                and X_before[j] == 1
                and force_on_vector[j] == 0
                and force_off_vector[j] == 0
            ):

                W_if_off = (
                    W_T_previous[j]
                    + 9.0 / T_OFF_now[j]
                )

                W_max_now = get_w_max(
                    j,
                    i,
                    variable_hysteresis,
                )

                prevent_off_vector[j] = int(
                    W_if_off
                    >=
                    W_max_now - 1e-9
                )

        # ========================================================
        # STEP 6 — UPDATE OPTIMIZATION MODEL
        # ========================================================

        m.P_lim.set_value(
            float(P_lim)
        )

        for j in range(N_DEV):
            m.time_flex_available[j] = int(
                time_flex_available_now[j]
            )
            m.prevent_off[j] = int(
                prevent_off_vector[j]
            )

            if (
                Type[j] == 2
                and time_flex_available_now[j]
            ):

                W_prev_j = float(
                    W_T_previous[j]
                )

                if not np.isfinite(W_prev_j):
                    raise ValueError(
                        f"Invalid active W_T at minute {i}, "
                        f"device {j}."
                    )

                T_ON_j = float(T_ON_now[j])
                T_OFF_j = float(T_OFF_now[j])

                if not (
                    np.isfinite(T_ON_j)
                    and np.isfinite(T_OFF_j)
                    and T_ON_j > 0
                    and T_OFF_j > 0
                ):
                    raise ValueError(
                        f"Invalid active T_ON/T_OFF at minute {i}, "
                        f"device {j}."
                    )

                W_if_off_j = (
                    W_prev_j
                    + 9.0 / T_OFF_j
                )

                W_if_on_j = (
                    W_prev_j
                    - 9.0 / T_ON_j
                )

                WT_DA_j = float(
                    WT_DA_target_vector[j]
                )

                if not np.isfinite(WT_DA_j):
                    raise ValueError(
                        f"Missing active Day-Ahead WT target "
                        f"at minute {i}, device {j}."
                    )

                m.W_DA[j] = WT_DA_j
                m.W_if_off[j] = W_if_off_j
                m.W_if_on[j] = W_if_on_j

            else:

                # Inclui:
                # - Type 0 e Type 1;
                # - Type 2 fora da janela operational.
                #
                # Nunca enviar NaN ao Pyomo.
                m.W_DA[j] = 0.0
                m.W_if_off[j] = 0.0
                m.W_if_on[j] = 0.0

            # Power-flexibility utility weight
            if Type[j] == 1:
                m.WP_DA[j] = float(
                    WP_DA_target_vector[j]
                )

            else:

                m.WP_DA[j] = 0.0

            # State before optimization
            m.X_in[j] = int(
                X_before[j]
            )

            m.Y_in[j] = float(
                Y_before[j]
            )

            # Operational constraints
            m.force_on[j] = int(
                force_on_vector[j]
            )

            m.force_off[j] = int(
                force_off_vector[j]
            )

            # Power hold
            m.count_T_Hold_Power[j] = int(
                count_vector[j]
            )

            # Solver initial values
            m.X_opt[j].value = int(
                X_before[j]
            )

            m.Y_opt[j].value = float(
                Y_before[j]
            )

        m.P_rlx.value = 0.0

        # ========================================================
        # STEP 7 — REBUILD DYNAMIC CONSTRAINTS
        # ========================================================

        if hasattr(m, "status_logic"):
            m.del_component(
                m.status_logic
            )

        if hasattr(
            m,
            "power_hold_constraint"
        ):
            m.del_component(
                m.power_hold_constraint
            )

        m.status_logic = pe.Constraint(
            m.J,
            rule=status_logic_rule
        )

        m.power_hold_constraint = pe.Constraint(
            m.J,
            rule=power_hold_rule
        )

        # ========================================================
        # STEP 8 — REAL-TIME OPTIMIZATION
        # ========================================================

        rt_solver_started = perf_counter()
        results = solver_utility.solve(
            m,
            tee=False,
            load_solutions=False
        )

        rt_solver_time_s[i] = perf_counter() - rt_solver_started

        termination = (
            results.solver.termination_condition
        )

        if (
            termination
            ==
            pe.TerminationCondition.optimal
        ):

            m.solutions.load_from(
                results
            )

        else:

            raise RuntimeError(
                f"Max Utility optimization failed "
                f"at minute {i}: "
                f"{termination}"
            )

        # ========================================================
        # STEP 9 — EXECUTED RT DECISION
        # ========================================================

        X_final = np.zeros(
            N_DEV,
            dtype=int
        )

        Y_final = np.ones(
            N_DEV,
            dtype=float
        )

        for j in range(N_DEV):

            X_final[j] = int(
                round(
                    pe.value(
                        m.X_opt[j]
                    )
                )
            )

            if X_final[j] == 1:

                Y_final[j] = float(
                    pe.value(
                        m.Y_opt[j]
                    )
                )

            else:

                Y_final[j] = 1.0

        # ========================================================
        # STEP 10 — UPDATE POWER HOLD
        # ========================================================

        count_vector = (
            update_power_hold_after_optimization(

                X_opt_rt_vector=X_final,

                Y_before_opt_rt_vector=Y_before,

                Y_opt_rt_vector=Y_final,

                count_vector=count_vector,

                Type=Type,

                TimeHoldPower=TimeHoldPower
            )
        )

        # ========================================================
        # STEP 11 — FINAL RT STATES
        # ========================================================

        P_rlx_vector[i] = float(
            pe.value(
                m.P_rlx
            )
        )

        P_final = compute_power_vector(

            X_vec=X_final,

            Power_max=Power_max,

            Y_vec=Y_final
        )

        W_T_final = update_time_weights(
            prev_W_T=W_T_previous,
            X_now=X_final,
            Type=Type,
            T_ON_now=T_ON_now,
            T_OFF_now=T_OFF_now,
            minute=i,
            time_flex_available=time_flex_available_now,
            variable_hysteresis=variable_hysteresis,
        )

        W_P_final = compute_wp_vector(

            Y_vec=Y_final,

            Weight_P_min=Weight_P_min,

            Weight_P_max=Weight_P_max,

            Y_min=Y_min,

            Y_max=Y_max
        )

        # ========================================================
        # STEP 12 — COMBINED FLEXIBILITY WEIGHT
        # ========================================================

        W_combined_final = np.full(
            N_DEV,
            np.nan,
            dtype=float
        )

        for j in range(N_DEV):

            if Type[j] == 1:
                W_combined_final[j] = W_P_final[j]

            elif Type[j] == 2:
                W_combined_final[j] = W_T_final[j]

        # ========================================================
        # STEP 13 — STORE RT STATE
        # ========================================================

        X_rt_matrix[i, :] = X_final

        Y_rt_matrix[i, :] = Y_final

        W_T_rt_matrix[i, :] = W_T_final

        W_P_rt_matrix[i, :] = W_P_final

        W_rt_matrix[i, :] = W_combined_final

        P_rt_matrix[i, :] = P_final

        P_rt_sum[i] = compute_power_sum(
            P_final
        )

        force_on_rt_matrix[
            i, :
        ] = force_on_vector

        force_off_rt_matrix[
            i, :
        ] = force_off_vector

        count_hold_matrix[
            i, :
        ] = count_vector

        rt_iteration_time_s[i] = perf_counter() - rt_iteration_started
else:
    print("Skipped: no feasible day-ahead solution.")


# Integrated real-time plots and metrics

In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    plot_stacked_device_power(
        X_matrix=X_onoff_FC_matrix,
        Y_matrix=Y_FC_matrix,
        Power_max=Power_max,
        dev_names=dev_names,

        time_start=0,
        time_finish=24*60,

        P_lim_vector=P_lim_vector,

        colors=colors,

        tick_step=60,

        title="Baseline"
    )
    plot_stacked_device_power(
        X_matrix=X_rt_matrix,
        Y_matrix=Y_rt_matrix,
        Power_max=Power_max,
        dev_names=dev_names,

        time_start=0,
        time_finish=24*60,

        P_lim_vector=P_lim_vector,

        colors=colors,

        tick_step=60,

        title="Max Utility"
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# A valid day-ahead solution is required for this cell.
if day_ahead_feasible:
    # Aggregate power trajectories used by every physical metric.
    P_baseline = np.sum(
        X_onoff_FC_matrix
        * Y_FC_matrix
        * Power_max.reshape(1, -1),
        axis=1,
    )
    P_rt = np.asarray(P_rt_sum, dtype=float)

    # Apply the same minute-level ToU tariff to both trajectories.
    minute_price_EUR_kWh = np.full(T, np.nan, dtype=float)
    for timezone, limits in TIME_ZONES.items():
        minute_price_EUR_kWh[
            limits["start"]:limits["end"]
        ] = TOU_PRICE_EUR_KWH[timezone]
    if not np.all(np.isfinite(minute_price_EUR_kWh)):
        raise ValueError("The ToU tariff does not cover the complete RT horizon.")

    def integrated_rt_physical_metrics(P):
        P = np.asarray(P, dtype=float).reshape(-1)
        if P.shape != P_lim_vector.shape or P.shape != minute_price_EUR_kWh.shape:
            raise ValueError("Power, limit and tariff vectors must have equal length.")
        exceedance_W = np.maximum(P - P_lim_vector, 0.0)
        violation_mask = exceedance_W > 1e-6
        return {
            "energy_kWh": float(P.sum() / (60.0 * 1000.0)),
            "cost_EUR": float(
                np.sum(minute_price_EUR_kWh * P / (60.0 * 1000.0))
            ),
            "violation_time_min": int(violation_mask.sum()),
            "mean_exceedance_kW": float(
                exceedance_W[violation_mask].mean() / 1000.0
                if violation_mask.any() else 0.0
            ),
            "max_exceedance_kW": float(exceedance_W.max() / 1000.0),
        }

    baseline_metrics = integrated_rt_physical_metrics(P_baseline)
    rt_metrics = integrated_rt_physical_metrics(P_rt)
    energy_reduction_pct = (
        100.0
        * (baseline_metrics["energy_kWh"] - rt_metrics["energy_kWh"])
        / baseline_metrics["energy_kWh"]
    )
    cost_reduction_pct = (
        100.0
        * (baseline_metrics["cost_EUR"] - rt_metrics["cost_EUR"])
        / baseline_metrics["cost_EUR"]
    )

    if (
        np.asarray(rt_iteration_time_s).shape != (T,)
        or np.asarray(rt_solver_time_s).shape != (T,)
        or not np.all(np.isfinite(rt_iteration_time_s))
        or not np.all(np.isfinite(rt_solver_time_s))
    ):
        raise ValueError("Incomplete integrated RT computation-time measurements.")

    scenario_label = 'DA Max Utility + RT'
    integrated_rt_metrics_df = pd.DataFrame([
        {
            "Group": "Energy",
            "Metric": "Total energy",
            "Unit": "kWh",
            "Baseline": baseline_metrics["energy_kWh"],
            scenario_label: rt_metrics["energy_kWh"],
        },
        {
            "Group": "Energy",
            "Metric": "Energy reduction",
            "Unit": "%",
            "Baseline": 0.0,
            scenario_label: energy_reduction_pct,
        },
        {
            "Group": "Cost",
            "Metric": "Total energy cost",
            "Unit": "EUR",
            "Baseline": baseline_metrics["cost_EUR"],
            scenario_label: rt_metrics["cost_EUR"],
        },
        {
            "Group": "Cost",
            "Metric": "Cost reduction",
            "Unit": "%",
            "Baseline": 0.0,
            scenario_label: cost_reduction_pct,
        },
        {
            "Group": "RT violations",
            "Metric": "Violation time",
            "Unit": "min",
            "Baseline": baseline_metrics["violation_time_min"],
            scenario_label: rt_metrics["violation_time_min"],
        },
        {
            "Group": "RT violations",
            "Metric": "Mean exceedance",
            "Unit": "kW",
            "Baseline": baseline_metrics["mean_exceedance_kW"],
            scenario_label: rt_metrics["mean_exceedance_kW"],
        },
        {
            "Group": "RT violations",
            "Metric": "Maximum exceedance",
            "Unit": "kW",
            "Baseline": baseline_metrics["max_exceedance_kW"],
            scenario_label: rt_metrics["max_exceedance_kW"],
        },
        {
            "Group": "Computation",
            "Metric": "DA planning time",
            "Unit": "s",
            "Baseline": np.nan,
            scenario_label: float(planning_time_s),
        },
        {
            "Group": "Computation",
            "Metric": "Mean RT iteration time",
            "Unit": "ms",
            "Baseline": np.nan,
            scenario_label: float(np.mean(rt_iteration_time_s) * 1000.0),
        },
        {
            "Group": "Computation",
            "Metric": "Maximum RT iteration time",
            "Unit": "ms",
            "Baseline": np.nan,
            scenario_label: float(np.max(rt_iteration_time_s) * 1000.0),
        },
    ])

    display(
        integrated_rt_metrics_df.style
        .hide(axis="index")
        .format(
            {
                "Baseline": lambda value: "--" if pd.isna(value) else f"{value:.2f}",
                scenario_label: lambda value: "--" if pd.isna(value) else f"{value:.2f}",
            }
        )
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# Reaggregate the minute-level RT trajectory into day-ahead blocks.
def summarize_rt_violations_by_block(
    P_rt,
    P_limit,
    block_size=10,
    tolerance=1e-6,
    da_blocks=None,
):
    """Compare minute-level RT violations with aggregate block-energy violations."""
    P_rt = np.asarray(P_rt, dtype=float).reshape(-1)
    P_limit = np.asarray(P_limit, dtype=float).reshape(-1)
    if P_rt.shape != P_limit.shape:
        raise ValueError("P_rt and P_limit must have the same shape.")
    if len(P_rt) % block_size != 0:
        raise ValueError("The RT horizon must be divisible by block_size.")

    rows = []
    for block, start in enumerate(range(0, len(P_rt), block_size)):
        end = start + block_size
        rt_block = P_rt[start:end]
        limit_block = P_limit[start:end]
        minute_excess = np.maximum(rt_block - limit_block, 0.0)
        rt_energy_Wh = float(rt_block.sum() / 60.0)
        limit_energy_Wh = float(limit_block.sum() / 60.0)
        block_excess_Wh = max(rt_energy_Wh - limit_energy_Wh, 0.0)
        rows.append({
            "Block": block,
            "Interval": f"{start // 60:02d}:{start % 60:02d}--{end // 60:02d}:{end % 60:02d}",
            "RT energy [Wh]": rt_energy_Wh,
            "Energy limit [Wh]": limit_energy_Wh,
            "RT block exceedance [Wh]": block_excess_Wh,
            "RT energy-violating block": block_excess_Wh > tolerance,
            "RT violation minutes": int((minute_excess > tolerance).sum()),
            "Maximum minute exceedance [W]": float(minute_excess.max()),
        })

    block_df = pd.DataFrame(rows)

    if da_blocks is not None and len(da_blocks) == len(block_df):
        da = da_blocks.sort_values("block").reset_index(drop=True)
        da_excess = (
            da["E_total_model_Wh"].to_numpy(dtype=float)
            - da["E_limit_Wh"].to_numpy(dtype=float)
        )
        block_df["DA energy-violating block"] = da_excess > tolerance

    has_minute_violation = block_df["RT violation minutes"] > 0
    has_energy_violation = block_df["RT energy-violating block"]
    minute_violations_in_compliant_blocks = int(
        block_df.loc[~has_energy_violation, "RT violation minutes"].sum()
    )

    summary = {
        "Total 10-min blocks": len(block_df),
        "RT violation minutes": int(block_df["RT violation minutes"].sum()),
        "Blocks with at least one RT violation minute": int(has_minute_violation.sum()),
        "Blocks whose RT energy exceeds E_lim": int(has_energy_violation.sum()),
        "RT violation minutes in energy-compliant blocks": minute_violations_in_compliant_blocks,
        "Total RT block exceedance [kWh]": float(
            block_df["RT block exceedance [Wh]"].sum() / 1000.0
        ),
    }
    if "DA energy-violating block" in block_df:
        summary["DA energy-violating blocks"] = int(
            block_df["DA energy-violating block"].sum()
        )
        summary["Blocks violated in both DA and RT energy"] = int(
            (block_df["DA energy-violating block"] & has_energy_violation).sum()
        )

    summary_df = pd.DataFrame([summary])
    detail_df = block_df.loc[
        has_minute_violation | has_energy_violation
    ].reset_index(drop=True)
    return summary_df, detail_df, block_df


if day_ahead_feasible:
    rt_block_violation_summary_df, rt_block_violation_detail_df, rt_blocks_df = (
        summarize_rt_violations_by_block(
            P_rt=P_rt_sum,
            P_limit=P_lim_vector,
            block_size=BLOCK_SIZE,
            tolerance=1e-6,
            da_blocks=energy_results_df,
        )
    )
    print("RT trajectory reaggregated into day-ahead blocks:")
    display(rt_block_violation_summary_df.round(6))
    print("Blocks containing minute-level or aggregate-energy violations:")
    display(rt_block_violation_detail_df.round(6))
else:
    print("Skipped: no feasible day-ahead solution.")


## Block-energy compliance versus minute-level power violations

The following figure selects an illustrative block in which both the day-ahead plan and the reaggregated real-time trajectory satisfy the block energy limit, although the real-time power exceeds the applicable limit during individual minutes.

In [ ]:
# Illustrate how an energy-compliant block can contain minute-level power violations.
def plot_energy_compliant_block_with_rt_violations(
    P_rt,
    P_baseline,
    P_limit,
    rt_blocks,
    da_blocks,
    block_size=10,
    tolerance=1e-6,
):
    """Plot all minutes of one energy-compliant block of the selected duration."""
    if not isinstance(block_size, (int, np.integer)) or block_size <= 0:
        raise ValueError("block_size must be a positive integer.")

    
    block= 43

    start = block * block_size
    end = start + block_size

    rt_power = np.asarray(P_rt, dtype=float)[start:end]
    baseline_power = np.asarray(P_baseline, dtype=float)[start:end]
    limit_power = np.asarray(P_limit, dtype=float)[start:end]
    minutes = np.arange(block_size)
    minute_excess = np.maximum(rt_power - limit_power, 0.0)
    before_violation_mask = baseline_power > limit_power + tolerance
    after_violation_mask = rt_power > limit_power + tolerance

    da = da_blocks.sort_values("block").reset_index(drop=True)
    da_energy_Wh = float(
        da.loc[da["block"] == block, "E_total_model_Wh"].iloc[0]
    )
    baseline_energy_Wh = float(baseline_power.sum() / 60.0)
    rt_energy_Wh = float(rt_power.sum() / 60.0)
    limit_energy_Wh = float(limit_power.sum() / 60.0)

    labels = [
        f"{(start + minute) // 60:02d}:{(start + minute) % 60:02d}"
        for minute in minutes
    ]
    figure_width = max(10.5, 0.42 * block_size)
    fig, (ax_power, ax_energy) = plt.subplots(
        2, 1, figsize=(figure_width, 7.0),
        gridspec_kw={"height_ratios": [1.45, 1.0]},
        constrained_layout=True,
    )

    # Each sample represents the complete interval [minute, minute + 1).
    # Using explicit bin edges avoids the half-minute displacement caused by
    # step="mid" and shades only the power above the limit.
    minute_edges = np.arange(block_size + 1)
    ax_power.stairs(
        baseline_power / 1000.0,
        minute_edges,
        color="#8C8C8C",
        linewidth=1.8,
        label="Baseline power",
    )
    ax_power.stairs(
        rt_power / 1000.0,
        minute_edges,
        color="#4F81BD",
        linewidth=2.0,
        label="RT power",
    )
    ax_power.stairs(
        limit_power / 1000.0,
        minute_edges,
        color="#C00000",
        linestyle="--",
        linewidth=1.8,
        label="Power limit",
    )
    marker_x = minutes + 0.5
    ax_power.scatter(
        marker_x[before_violation_mask],
        baseline_power[before_violation_mask] / 1000.0,
        color="red",
        marker="x",
        s=55,
        linewidths=1.8,
        label="Violation before RT control",
        zorder=6,
    )
    ax_power.scatter(
        marker_x[after_violation_mask],
        rt_power[after_violation_mask] / 1000.0,
        facecolors="none",
        edgecolors="purple",
        marker="o",
        s=105,
        linewidths=1.8,
        label="Violation after RT control",
        zorder=7,
    )
    ax_power.set_ylabel("Power [kW]")
    ax_power.set_xlim(0, block_size)
    ax_power.set_xticks(minutes + 0.5, labels, rotation=45, ha="right")
    ax_power.set_xlabel("Time")
    ax_power.grid(axis="y", alpha=0.25)
    ax_power.legend(loc="lower left", ncol=3, frameon=True)
    ax_power.set_title(
        f"Energy-compliant {block_size}-minute block with minute-level power violations"
    )

    energy_labels = ["Baseline", "DA planned", "RT executed"]
    energy_values = [baseline_energy_Wh, da_energy_Wh, rt_energy_Wh]
    energy_colors = ["#8C8C8C", "#70AD47", "#4F81BD"]
    bars = ax_energy.bar(
        energy_labels,
        energy_values,
        color=energy_colors,
        width=0.62,
        zorder=3,
    )
    ax_energy.axhline(
        limit_energy_Wh,
        color="#C00000",
        linestyle="--",
        linewidth=1.8,
        label=rf"Block energy limit ({limit_energy_Wh:.1f} Wh)",
        zorder=4,
    )
    ax_energy.bar_label(bars, fmt="%.1f Wh", padding=3, fontsize=9)
    ax_energy.set_ylabel("Total block energy [Wh]")
    ax_energy.set_title(
        f"Aggregate energy over the same {block_size}-minute block"
    )
    ax_energy.grid(axis="y", alpha=0.25)
    ax_energy.set_axisbelow(True)
    ax_energy.legend(loc="lower left", frameon=True)
    ax_energy.set_ylim(
        0,
        max(max(energy_values), limit_energy_Wh) * 1.16,
    )


    selected_data = pd.DataFrame({
        "Minute": labels,
        "Baseline power [W]": baseline_power,
        "RT power [W]": rt_power,
        "Power limit [W]": limit_power,
        "Exceedance [W]": minute_excess,
        "Violation before RT": before_violation_mask,
        "Violation after RT": after_violation_mask,
    })


    return fig, (ax_power, ax_energy), selected_data, block


if day_ahead_feasible:
    (
        rt_resolution_figure,
        rt_resolution_axes,
        rt_resolution_block_df,
        rt_resolution_block,
    ) = plot_energy_compliant_block_with_rt_violations(
        P_rt=P_rt_sum,
        P_baseline=P_baseline,
        P_limit=P_lim_vector,
        rt_blocks=rt_blocks_df,
        da_blocks=energy_results_df,
        block_size=BLOCK_SIZE,
        tolerance=1e-6,
    )
    display(rt_resolution_block_df.round(3))
    display(rt_resolution_figure)
    plt.close(rt_resolution_figure)
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# Drinks Refrigerator thermal reconstruction — integrated Max Utility
FRIDGE_INDEX = 0

P_THERMAL = 60.0       # W
MC_THERMAL = 29.66     # Wh/K
A_THERMAL = 3.21       # W/K
ETA_THERMAL = 3.0      # dimensionless
T_AMBIENT = 18.0       # degrees C
T_INITIAL = 4.0        # degrees C
H_MINUS = 2.0          # degrees C
H_PLUS = 6.0           # degrees C
DT_MINUTES = 1


def reconstruct_fridge_temperature(
    X,
    mc=MC_THERMAL,
    T0=T_INITIAL,
    P=P_THERMAL,
    A=A_THERMAL,
    eta=ETA_THERMAL,
    Tamb=T_AMBIENT,
    dt_minutes=DT_MINUTES,
):
    """Reconstruct refrigerator temperature from its compressor state."""
    X = np.asarray(X, dtype=int)
    if X.ndim != 1:
        raise ValueError("X must be a one-dimensional activation sequence.")
    if not np.all(np.isin(X, [0, 1])):
        raise ValueError("X must contain only binary values 0 and 1.")

    T_temperature = np.zeros(len(X), dtype=float)
    T_temperature[0] = T0
    dt_hours = dt_minutes / 60.0
    epsilon = np.exp(-(dt_hours * A) / mc)

    for minute in range(len(X) - 1):
        compressor_power = P * X[minute]
        T_temperature[minute + 1] = (
            epsilon * (T_temperature[minute] - Tamb)
            + Tamb
            - eta * (compressor_power / A) * (1.0 - epsilon)
        )

    return T_temperature


def plot_fridge_temperature_smartfridge(
    T_baseline,
    T_opt,
    X_baseline,
    X_opt,
    objective_name,
    SP=4.0,
    H_minus=H_MINUS,
    H_plus=H_PLUS,
    tick_step=60,
):
    n = len(T_opt)
    x = np.arange(n)
    labels = [f"{minute // 60:02d}:{minute % 60:02d}" for minute in x]

    fig, ax_temperature = plt.subplots(figsize=(18, 6))
    ax_temperature.plot(
        x, T_baseline, linewidth=3, color="#4F81BD",
        label="Temperature baseline",
    )
    ax_temperature.plot(
        x, T_opt, linewidth=3, color="#F79646",
        label=f"Temperature integrated {objective_name}",
    )
    ax_temperature.axhline(
        SP, linestyle=":", linewidth=2.5, color="black", label="Setpoint",
    )
    ax_temperature.axhline(
        H_minus, linestyle="--", linewidth=2, color="gray", label="H- lower limit",
    )
    ax_temperature.axhline(
        H_plus, linestyle="--", linewidth=2, color="gray", label="H+ upper limit",
    )
    ax_temperature.set_xlabel("Time", fontsize=18, fontweight="bold")
    ax_temperature.set_ylabel("Temperature [°C]", fontsize=18, fontweight="bold")
    ax_temperature.set_xticks(np.arange(0, n, tick_step))
    ax_temperature.set_xticklabels(labels[::tick_step], rotation=90, fontsize=11)
    ax_temperature.grid(axis="y", alpha=0.3)

    ax_status = ax_temperature.twinx()
    ax_status.step(
        x, X_baseline, where="post", linewidth=3, color="#4F81BD",
        alpha=0.30, label="X baseline",
    )
    ax_status.step(
        x, X_opt, where="post", linewidth=3, color="#F79646",
        alpha=0.30, label=f"X integrated {objective_name}",
    )
    ax_status.set_ylabel("Compressor status", fontsize=18, fontweight="bold")
    ax_status.set_ylim(-0.1, 1.1)
    ax_status.set_yticks([0, 1])

    lines_temperature, labels_temperature = ax_temperature.get_legend_handles_labels()
    lines_status, labels_status = ax_status.get_legend_handles_labels()

    # Arrange the legend in three visual groups. The two invisible
    # entries keep the setpoint centred between the thermal limits.
    from matplotlib.lines import Line2D
    blank = Line2D([], [], linestyle="none", label="")
    legend_lines = [
        lines_temperature[0], lines_temperature[1], blank,
        lines_temperature[3], lines_temperature[2], lines_temperature[4],
        lines_status[0], lines_status[1], blank,
    ]
    legend_labels = [
        labels_temperature[0], labels_temperature[1], "",
        labels_temperature[3], labels_temperature[2], labels_temperature[4],
        labels_status[0], labels_status[1], "",
    ]
    ax_temperature.legend(
        legend_lines,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        frameon=False,
        fontsize=15,
    )
    plt.tight_layout()


    plt.show()
    return fig, (ax_temperature, ax_status)


if day_ahead_feasible:
    X_fridge_baseline = np.asarray(
        X_onoff_FC_matrix[:, FRIDGE_INDEX], dtype=int
    )
    X_fridge_integrated = np.asarray(
        X_rt_matrix[:, FRIDGE_INDEX], dtype=int
    )

    T_baseline = reconstruct_fridge_temperature(
        X=X_fridge_baseline,
        mc=MC_THERMAL,
    )
    T_integrated_max_utility = reconstruct_fridge_temperature(
        X=X_fridge_integrated,
        mc=MC_THERMAL,
    )

    fridge_temperature_figure, fridge_temperature_axes = (
        plot_fridge_temperature_smartfridge(
            T_baseline=T_baseline,
            T_opt=T_integrated_max_utility,
            X_baseline=X_fridge_baseline,
            X_opt=X_fridge_integrated,
            objective_name="Max Utility",
            SP=4.0,
            H_minus=H_MINUS,
            H_plus=H_PLUS,
            tick_step=60,
        )
    )
else:
    print("Skipped: no feasible day-ahead solution.")


In [ ]:
# Reconstructed refrigerator temperature metrics
def compute_fridge_temperature_metrics(
    T_baseline,
    T_integrated,
    objective_name,
    SP=4.0,
):
    """Use the same temperature metrics as the standalone RT analysis."""
    import numpy as np
    import pandas as pd

    cases = {
        "Baseline": np.asarray(T_baseline, dtype=float),
        f"DA + RT - {objective_name}": np.asarray(T_integrated, dtype=float),
    }

    rows = []
    for name, T in cases.items():
        rows.append({
            "Case": name,
            "Mean temperature [°C]": float(np.mean(T)),
            "Max temperature [°C]": float(np.max(T)),
            "Min temperature [°C]": float(np.min(T)),
            "Mean abs. deviation from SP [°C]": float(np.mean(np.abs(T - SP))),
        })

    return pd.DataFrame(rows)


if day_ahead_feasible:
    fridge_temperature_metrics = compute_fridge_temperature_metrics(
        T_baseline=T_baseline,
        T_integrated=T_integrated_max_utility,
        objective_name="Max Utility",
        SP=4.0,
    )
    display(fridge_temperature_metrics)
else:
    fridge_temperature_metrics = None
    print("Skipped: no feasible day-ahead solution.")


## Baseline consumption in 30-minute blocks

In [ ]:
def plot_thesis_baseline_consumption():
    """Display the thesis baseline in 30-minute blocks, without exports."""
    block_size = 30
    power_W = X_onoff_FC_matrix * Y_FC_matrix * Power_max
    assert power_W.shape[0] == 1440 and np.isfinite(power_W).all()
    energy = power_W.reshape(48, block_size, -1).sum(axis=1) / 60
    np.testing.assert_allclose(energy.sum(), power_W.sum() / 60)
    limits_W = np.zeros(1440)
    for tz, interval in TIME_ZONES.items():
        limits_W[interval["start"]:interval["end"]] = TIMEZONE_POWER_LIMITS_W[tz]
    limits = limits_W.reshape(48, block_size).sum(axis=1) / 60
    starts = np.arange(48) / 2
    fig, ax = plt.subplots(figsize=(15, 6))
    bottom = np.zeros(48)
    for j, name in enumerate(dev_names):
        ax.bar(starts, energy[:, j], bottom=bottom, width=0.5, align="edge",
               color=colors[j], label=name, linewidth=0)
        bottom += energy[:, j]
    ax.stairs(limits, np.arange(49) / 2, baseline=None, color="black",
              linestyle="--", linewidth=1.8, label="Block energy limit")
    ax.set(xlim=(0, 24), ylim=(0, None), xlabel="Time of day",
           ylabel="Energy consumption per 30-minute block [Wh]",
           title="Baseline consumption — 30-minute blocks")
    ax.set_xticks(np.arange(0, 25, 2), [f"{h:02d}:00" for h in range(0, 25, 2)])
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
    fig.tight_layout()
    return fig, ax

plot_thesis_baseline_consumption()
plt.show()


## Optional block-size selection experiment

The thesis figure `block_size_selection_status_table.pdf` is generated by the functions below. Set `RUN_BLOCK_SIZE_COMPARISON = True` to recompute B=10, 15 and 30 using the current notebook settings. This can take about 12 minutes; no historical result files are required.

In [ ]:
from matplotlib.patches import Rectangle
BLOCK_SIZES = (10, 15, 30)

def _with_reductions(data):
    data = data.copy()
    data["violation_reduction_pct"] = 100.0 * (
        1.0 - data["plan_violation_kWh"] / data["baseline_violation_kWh"]
    )
    data["consumption_reduction_pct"] = 100.0 * (
        1.0 - data["plan_energy_kWh"] / data["baseline_energy_kWh"]
    )
    return data

def comparison_status_table(results):
    """Build the compact thesis table and retain solver status per B."""
    results = _with_reductions(results)
    rows = []
    for scenario in ("Max Utility", "Min Consumption", "Min Cost"):
        selected = results.loc[results["scenario"] == scenario].set_index("block_size_min")
        row = {"Scenario": scenario}
        for block_size in BLOCK_SIZES:
            row[f"Gap B={block_size} [%]"] = selected.loc[block_size, "mip_gap_pct"]
        for block_size in BLOCK_SIZES:
            row[f"Violation reduction B={block_size} [%]"] = selected.loc[
                block_size, "violation_reduction_pct"
            ]
        for block_size in BLOCK_SIZES:
            row[f"Consumption reduction B={block_size} [%]"] = selected.loc[
                block_size, "consumption_reduction_pct"
            ]
        rows.append(row)
    return pd.DataFrame(rows), results

def plot_gap_violation_status_table(results):
    """Plot the three selection criteria, colored by solver status."""
    table, results = comparison_status_table(results)
    columns = ["", "B=10", "B=15", "B=30", "B=10", "B=15", "B=30",
               "B=10", "B=15", "B=30"]
    cell_text = []
    for _, row in table.iterrows():
        values = [row["Scenario"]]
        for block_size in BLOCK_SIZES:
            value = row[f"Gap B={block_size} [%]"]
            values.append("—" if pd.isna(value) else f"{value:.3f}")
        for block_size in BLOCK_SIZES:
            values.append(f"{row[f'Violation reduction B={block_size} [%]']:.2f}")
        for block_size in BLOCK_SIZES:
            values.append(f"{row[f'Consumption reduction B={block_size} [%]']:.2f}")
        cell_text.append(values)

    fig, ax = plt.subplots(figsize=(18, 4.8))
    ax.axis("off")
    table_bbox = [0.01, 0.05, 0.98, 0.70]
    col_widths = [0.18] + [(1.0 - 0.18) / 9] * 9
    visual = ax.table(
        cellText=cell_text, colLabels=columns, cellLoc="center",
        colWidths=col_widths, bbox=table_bbox,
    )
    visual.auto_set_font_size(False)
    visual.set_fontsize(13.0)
    header = "#D9E2F3"
    optimal = "#C6EFCE"
    time_limited = "#FCE4B2"
    for col in range(len(columns)):
        visual[(0, col)].set_facecolor(header)
        visual[(0, col)].set_text_props(weight="bold")

    # First header level: one label spanning the three B columns of each metric.
    x0, _, table_width, table_height = table_bbox
    group_y, group_height = 0.75, 0.17
    groups = ((0, 1, "Scenario"), (1, 3, "Gap [%]"),
              (4, 3, "Violation reduction [%]"),
              (7, 3, "Consumption reduction [%]"))
    for start, span, label in groups:
        left = x0 + table_width * sum(col_widths[:start])
        width = table_width * sum(col_widths[start:start + span])
        bottom = group_y
        height = group_height
        if label == "Scenario":
            bottom -= table_height / 4
            height += table_height / 4
        ax.add_patch(Rectangle(
            (left, bottom), width, height, transform=ax.transAxes,
            facecolor=header, edgecolor="black", linewidth=1.0,
        ))
        ax.text(
            left + width / 2, bottom + height / 2, label,
            transform=ax.transAxes, ha="center", va="center",
            fontsize=14.0, fontweight="bold",
        )
    scenario_order = ["Max Utility", "Min Consumption", "Min Cost"]
    for row_index, scenario in enumerate(scenario_order, start=1):
        visual[(row_index, 0)].set_text_props(weight="bold", ha="left")
        selected = results.loc[results["scenario"] == scenario].set_index("block_size_min")
        for b_index, block_size in enumerate(BLOCK_SIZES):
            color = optimal if bool(selected.loc[block_size, "optimality_proven"]) else time_limited
            # Apply the solve status to all metrics obtained from this run.
            visual[(row_index, 1 + b_index)].set_facecolor(color)
            visual[(row_index, 4 + b_index)].set_facecolor(color)
            visual[(row_index, 7 + b_index)].set_facecolor(color)
    settings = results[["time_limit_s", "mip_gap_target_pct", "lambda_rlx"]].drop_duplicates()
    if len(settings) != 1:
        raise ValueError("Cannot label a comparison with mixed solver parameters")
    limit, tolerance, penalty = settings.iloc[0]
    ax.text(
        0.5, 0.94,
        f"Time limit: {limit:g} s per optimization   |   MIP-gap tolerance: {tolerance:g}%   |   "
        + rf"$\mathbf{{\lambda_{{rlx}} = {penalty:g}}}$",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=14, fontweight="bold",
    )
    ax.text(
        0.5, -0.03,
        f"Green: optimal within {tolerance:g}% tolerance   |   Orange: {limit:g} s time limit"
        "   |   Max Utility: optimization 2; others: single optimization",
        transform=ax.transAxes, ha="center", va="top", fontsize=12,
    )
    return fig, ax

def run_block_size_comparison():
    """Recompute the thesis selection table (12 solves, up to 60 s each)."""
    rows = []
    for strategy, notebook in {
        "Max Utility": "Day_Ahead_Plan_Max_Utility.ipynb",
        "Min Consumption": "Day_Ahead_Plan_Min_Consumption.ipynb",
        "Min Cost": "Day_Ahead_Plan_Min_Cost.ipynb",
    }.items():
        for block_size in BLOCK_SIZES:
            scope = run_day_ahead_for_comparison(notebook, block_size)
            energy = scope["energy_results_df"]
            diagnostic = scope["final_diagnostics"]
            rows.append({
                "scenario": strategy, "block_size_min": block_size,
                "mip_gap_pct": diagnostic["mip_gap_pct"],
                "optimality_proven": diagnostic["optimality_proven"],
                "time_limit_s": diagnostic["time_limit_s"],
                "mip_gap_target_pct": 100 * diagnostic["mip_rel_gap_target"],
                "lambda_rlx": scope["lambda_rlx"],
                "baseline_energy_kWh": energy["E_baseline_Wh"].sum() / 1000,
                "plan_energy_kWh": energy["E_total_model_Wh"].sum() / 1000,
                "baseline_violation_kWh": (energy["E_baseline_Wh"] - energy["E_limit_Wh"]).clip(lower=0).sum() / 1000,
                "plan_violation_kWh": (energy["E_total_model_Wh"] - energy["E_limit_Wh"]).clip(lower=0).sum() / 1000,
            })
    return pd.DataFrame(rows)

# Enable explicitly when a new block-size experiment is required.
RUN_BLOCK_SIZE_COMPARISON = False
if RUN_BLOCK_SIZE_COMPARISON:
    block_size_results = run_block_size_comparison()
    plot_gap_violation_status_table(block_size_results)
    plt.show()
